# Semanas 5-6 — WVC 2026 — fechamento experimental

**Escopo (§11 do protocolo):** rodar o fatorial-núcleo, análise por classe (H4), matrizes de confusão — **e**, porque o calendário não permite mais uma iteração, **o toque único no 7k**, que é o que efetivamente produz as tabelas do paper.

**Restrição dura desta fase `[declarada]`:** as Semanas 3-4 **não podem ser re-rodadas**. Isso não é um detalhe operacional — ele *fecha* três decisões que os registros anteriores deixaram abertas, e o fechamento é para o lado conservador:

| Item aberto (registro 3-4, §8) | Decisão forçada aqui | Consequência para o texto |
|---|---|---|
| Re-rodar ablação H3-a só-cor (mantendo crop) | **Impossível** → saída (2) do §3 | H3-a **rebaixada** a achado direcional com limitação declarada. **Não** pode ser contribuição central. |
| Re-treinar SimCLR com timebox maior | **Impossível** | SimCLR declarado **budget-limited, não-convergido**; o gap Phikon−SimCLR é **limite superior**. |
| Pré-treinar em 100k para o número-manchete | **Impossível** | Manchete sai de SSL treinado em **80k** (train_pool). Declarar; é conservador (menos dados, não mais). |

Isso é bom para a honestidade do paper e ruim para a força de H3-a. A contribuição passa a se apoiar em: **(i)** a barra ordenada clássico < SSL < FM no 7k, **(ii)** a refutação de H2 com mecanismo de densidade de informação, **(iii)** H4 (decomposição por classe) e **(iv)** o lado clássico *enriquecido* (H3-b) — que é o que este notebook ainda pode mover.

---

## O que este notebook faz que os anteriores não fizeram

1. **H3-b de verdade** (o único braço ainda melhorável sem GPU): transformada de Hellinger, LBP-HF (preserva orientação), LBP de cor (por canal + *opponent-color*), e o **descritor fractal volumétrico Bouligand–Minkowski + multifractal** — a resposta direta ao risco de venue do §4-b da Semana 2.
2. **H4** (pré-registrada): recall por classe, matrizes de confusão, e o teste direcional *morfológicas vs. texturais* com IC por bootstrap.
3. **O toque único no 7k**, com **pré-commit hasheado**: todos os braços × {RAW, PCA32} × 5 frações × 5 seeds, em **uma** execução, com predições salvas para estatística pareada posterior sem refit.
4. **Estatística que sobrevive a revisor**: diferenças pareadas com IC por bootstrap sobre o conjunto de teste (o único IC honesto em `frac=100%`, onde só há 1 seed efetiva), tamanho de efeito, correção de Holm.
5. **Confirmações pendentes**: stain (Macenko), alinhamento dos caches herdados, e a bibliografia do eva.


## O que este notebook NÃO prova (ler antes de escrever)

Herdado + novo:

- **(a)** Números **internos** continuam sendo diagnóstico, nunca tabela de desempenho. Só o 7k entra. Mantido.
- **(b)** **H3-a é colapso de pretexto**, não ablação de invariância (§3 do registro 3-4). Este notebook **não conserta isso** e não tenta: as features abladas passam pelo probe aqui só para que o achado direcional seja reportado com o mesmo aparato dos demais — **se** os checkpoints ablados existirem. A limitação declarada é obrigatória.
- **(c)** O SimCLR é **piso, não teto** (subtreinado, parado por relógio na subida).
- **(d)** **H3-b muda o lado clássico, não o aprendido.** Se o gap encolher, isso *não* significa "clássico alcança"; significa "parte do gap era pobreza do descritor, não do paradigma". Escrever exatamente assim.
- **(e) `[novo, crítico]` O Wilcoxon pareado do §7 do protocolo é inviável com 5 seeds.** Com n=5, o menor p bilateral do Wilcoxon com sinais é **0,0625** — nunca cruza 0,05, por construção. Reportá-lo como teste principal seria um erro auto-infligido. **Substituição registrada:** diferença pareada + **IC por bootstrap sobre o conjunto de teste 7k** (válido inclusive em frac=100%, onde não há seeds) + tamanho de efeito; Wilcoxon unilateral só como suplemento.
- **(f) `[novo]` A pré-registração de H4 cobre 6 das 9 classes.** O §2 nomeia morfológicas {TUM, NORM} e texturais {STR, MUS, ADI, MUC}. **BACK, DEB e LYM não foram classificadas a priori** — entram na tabela por completude, mas **fora** do teste pré-registrado. Não realocá-las depois de ver o resultado.


In [ ]:
# ============================================================================
# 0. CONFIGURACAO (edite so esta celula)
# ============================================================================
import os
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")   # herdado: evita download corrompido do Phikon
from pathlib import Path
import numpy as np

# --- Dados (mesma condicao/ordem de todas as semanas) ------------------------
NCT_DIR     = Path("./NCT-CRC-HE-100K")   # <-- EDITE
CRC_VAL_DIR = Path("./CRC-VAL-HE-7K")     # <-- EDITE

# --- Caches herdados (somente leitura) ---------------------------------------
WEEK2_CACHE_DIR  = Path("./cache_semana2")    # lbp_100k / frac_100k / phikon_100k
WEEK34_CACHE_DIR = Path("./cache_semana34")   # simclr_s{0,1,2}_full_100k
CKPT_DIR         = Path("./ckpt_simclr")      # simclr_s{seed}_{aug}_best.pt

# --- Saidas desta fase --------------------------------------------------------
CACHE_DIR = Path("./cache_semana56"); CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR   = Path("./out_semana56");   OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR   = OUT_DIR / "figs";         FIG_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR  = OUT_DIR / "preds_7k";     PRED_DIR.mkdir(parents=True, exist_ok=True)

# --- Protocolo TRAVADO (herdado do gate) -------------------------------------
INTERNAL_SPLIT_SEED = 20260816
INTERNAL_TEST_SIZE  = 0.20
CLASSES = ["ADI", "BACK", "DEB", "LYM", "MUC", "MUS", "NORM", "STR", "TUM"]

# H4 pre-registrada (§2). BACK/DEB/LYM NAO foram classificadas a priori.
H4_MORFO    = ["TUM", "NORM"]
H4_TEXTURAL = ["STR", "MUS", "ADI", "MUC"]
H4_NAO_PREREG = ["BACK", "DEB", "LYM"]

LABEL_FRACTIONS = [0.01, 0.05, 0.10, 0.25, 1.00]
LABEL_SEEDS     = [0, 1, 2, 3, 4]
COMMON_D        = 32
D_SWEEP         = [2, 4, 8, 16, 24, 32, 48, 64]

# --- SSL herdado --------------------------------------------------------------
SIMCLR_SEEDS      = [0, 1, 2]
SIMCLR_AUG        = "full"
SIMCLR_AUG_ABL    = "ablated_no_scale_color"
SIMCLR_BACKBONE   = "resnet50"
SIMCLR_PROJ_DIM   = 128
HEADLINE_SSL_SEED = 0          # braco SSL da tabela manchete

# --- Custo / paralelismo ------------------------------------------------------
# Respeita a alocacao do SLURM: nunca mais processos que CPUs pedidas no --cpus-per-task.
N_WORKERS   = int(os.environ.get("SLURM_CPUS_PER_TASK", 8))
CHUNK       = 256
PROBE_MAX_ITER = 2000
PROBE_N_JOBS   = 1             # lbfgs multinomial ignora n_jobs; -1 so causa oversubscription
BOOTSTRAP_B    = 2000

# --- Knobs de custo dos descritores novos ------------------------------------
USE_OPPONENT_COLOR = True      # opponent-color LBP (dobra o custo do braco de cor)
BM_S, BM_Z = 64, 64            # volume de Bouligand-Minkowski (S x S x Z)

# --- Guarda de toque unico ----------------------------------------------------
PRECOMMIT_PATH = OUT_DIR / "precommit_7k.json"
SEVENK_LOG     = OUT_DIR / "resultados_7k.jsonl"

GLOBAL_SEED = 20260818
np.random.seed(GLOBAL_SEED)
print("Config OK |", "workers:", N_WORKERS, "| opponent:", USE_OPPONENT_COLOR,
      "| BM:", (BM_S, BM_Z))
print("threads BLAS: OMP_NUM_THREADS =", os.environ.get("OMP_NUM_THREADS", "<nao setado>"),
      "| SLURM_CPUS_PER_TASK =", os.environ.get("SLURM_CPUS_PER_TASK", "<fora do SLURM>"))
if os.environ.get("OMP_NUM_THREADS") is None:
    print("  [AVISO] sem OMP_NUM_THREADS, o BLAS abre uma thread por nucleo DA MAQUINA, nao da")
    print("  alocacao. Com 8 CPUs alocadas e 8 processos de extracao isso derruba a vazao.")
    print("  Exporte OMP_NUM_THREADS=1 no job de extracao e =$SLURM_CPUS_PER_TASK no de probes.")

In [ ]:
# ============================================================================
# Imports
# ============================================================================
import json, hashlib, time, warnings, math, itertools, sys
import numpy as np
import pandas as pd
from PIL import Image
from concurrent.futures import ProcessPoolExecutor

from skimage.feature import local_binary_pattern
from scipy.ndimage import uniform_filter, distance_transform_edt, map_coordinates
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import balanced_accuracy_score, f1_score, confusion_matrix

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
print("libs ok | numpy", np.__version__, "| sklearn", __import__("sklearn").__version__)

In [ ]:
# ============================================================================
# 1. Indexacao + split interno (identicos ao gate / Semana 2 / Semanas 3-4)
# ============================================================================
def index_folder(root: Path, classes):
    paths, y = [], []
    for ci, c in enumerate(classes):
        cdir = root / c
        if not cdir.is_dir():
            raise FileNotFoundError(f"Subpasta de classe ausente: {cdir}")
        files = sorted(p for p in cdir.iterdir()
                       if p.suffix.lower() in {".tif", ".tiff", ".png", ".jpg", ".jpeg"})
        paths += files; y += [ci] * len(files)
    return np.array([str(p) for p in paths]), np.array(y, dtype=np.int64)

paths_100k, y_100k = index_folder(NCT_DIR, CLASSES)
N = len(paths_100k)
paths_7k, y_7k = index_folder(CRC_VAL_DIR, CLASSES)
N7 = len(paths_7k)
print(f"100k: {N} imagens | 7k: {N7} imagens")
print("100k por classe:", dict(zip(CLASSES, np.bincount(y_100k, minlength=9))))
print("7k  por classe:", dict(zip(CLASSES, np.bincount(y_7k, minlength=9))))

idx_all = np.arange(N)
idx_train_pool, idx_eval_int = train_test_split(
    idx_all, test_size=INTERNAL_TEST_SIZE, stratify=y_100k, random_state=INTERNAL_SPLIT_SEED)
idx_train_pool = np.sort(idx_train_pool); idx_eval_int = np.sort(idx_eval_int)
y_train_pool = y_100k[idx_train_pool]
print(f"Split interno: train_pool={len(idx_train_pool)} eval_int={len(idx_eval_int)}")
print("  (gate deu 80002/19998; Semana 2 deu 80000/20000 - realizacao varia com a versao da lib.")
print("   Registrar o valor DESTA execucao; numeros sao comparaveis, nao intercambiaveis celula-a-celula.)")

def label_subsample(frac, seed):
    if frac >= 1.0:
        return np.arange(len(idx_train_pool))
    sub, _ = train_test_split(np.arange(len(idx_train_pool)), train_size=frac,
                              stratify=y_train_pool, random_state=seed)
    return np.sort(sub)

_a = label_subsample(0.05, 3); _b = label_subsample(0.05, 3)
assert np.array_equal(_a, _b) and not np.array_equal(_a, label_subsample(0.05, 4))
print("Subamostragem deterministica e seed-sensivel: OK")

## 2. Confirmação de stain `[item aberto desde a Semana 2 — fechado aqui]`

Duas coisas, uma documental e uma empírica.

**Documental.** O registro Zenodo 10.5281/zenodo.1214456 descreve `NCT-CRC-HE-100K` como **normalizado por Macenko**, e distribui separadamente `NCT-CRC-HE-100K-NONORM` (mesma matéria-prima, **sem** normalização — e, importante, com *tiles não idênticos*, porque a seleção de patches foi estocástica). Portanto: se a pasta apontada por `NCT_DIR` veio do arquivo `NCT-CRC-HE-100K.zip`, a condição do núcleo (§3, "apenas Macenko") **bate com o declarado** e o item fecha. Se veio do `-NONORM`, o protocolo declarado está errado — não a análise.

**Empírica `[o que a célula abaixo mede]`.** O ponto que ninguém checou até agora e que é mais perigoso: **o 7k é a mesma condição de stain que o 100k?** A descrição oficial afirma a normalização explicitamente para o 100k; para o CRC-VAL-HE-7K a redação é menos explícita. Se as duas coortes diferirem em stain, parte dos ~6,8 pts de queda coorte-interna→7k medidos no gate é **domain shift de cor**, não dificuldade de coorte — e isso afeta os braços de forma **assimétrica**: LBP/Fractal são *grayscale* (quase imunes), Phikon e SimCLR são RGB (expostos), e o LBP-de-cor que este notebook introduz passa a ser exposto também.

A célula mede densidade óptica média e dispersão por canal nas duas pastas. Não conserta nada — **produz a frase que vai para a seção de limitações**, com número.

In [ ]:
# ============================================================================
# 2. Diagnostico de stain (100k vs 7k)
# ============================================================================
def od_stats(paths, n=500, seed=0):
    rng = np.random.default_rng(seed)
    sel = rng.choice(len(paths), size=min(n, len(paths)), replace=False)
    M = np.empty((len(sel), 3), dtype=np.float64)
    for i, p in enumerate(paths[sel]):
        a = np.asarray(Image.open(p).convert("RGB"), dtype=np.float32) / 255.0
        od = -np.log(np.clip(a, 1e-3, 1.0))
        M[i] = od.reshape(-1, 3).mean(axis=0)
    return M

M100 = od_stats(paths_100k, n=500, seed=1)
M7   = od_stats(paths_7k,   n=500, seed=1)

def _fmt(M, nome):
    mu, sd = M.mean(0), M.std(0)
    ratio = M / np.maximum(M.sum(1, keepdims=True), 1e-9)
    print(f"{nome:6s} OD medio R/G/B = {mu[0]:.3f}/{mu[1]:.3f}/{mu[2]:.3f}   "
          f"desvio entre imagens = {sd[0]:.3f}/{sd[1]:.3f}/{sd[2]:.3f}   "
          f"razao cromatica media = {ratio.mean(0).round(3)}")
    return mu, sd, ratio.mean(0)

mu1, sd1, r1 = _fmt(M100, "100k")
mu7, sd7, r7 = _fmt(M7,   "7k")

delta_chroma = float(np.abs(r1 - r7).sum())
ratio_disp   = float((sd7 / np.maximum(sd1, 1e-9)).mean())
print(f"\n|delta| de razao cromatica 100k vs 7k = {delta_chroma:.4f}")
print(f"dispersao relativa (7k / 100k)        = {ratio_disp:.2f}x")
print("\nLeitura (heuristica, NAO substitui a checagem documental):")
print("  delta_chroma < 0.02 e dispersao ~1x  -> mesma condicao de stain; item fechado.")
print("  delta_chroma > 0.05 OU dispersao > 1.5x -> coortes em condicoes DIFERENTES:")
print("    declarar como limitacao e notar que bracos grayscale (LBP/Fractal) sao menos")
print("    expostos que bracos RGB (Phikon/SimCLR/LBP-cor) - isso muda a leitura do gap no 7k.")
STAIN_DIAG = dict(od_100k=mu1.round(4).tolist(), od_7k=mu7.round(4).tolist(),
                  sd_100k=sd1.round(4).tolist(), sd_7k=sd7.round(4).tolist(),
                  delta_chroma=round(delta_chroma, 4), disp_rel=round(ratio_disp, 3))
json.dump(STAIN_DIAG, open(OUT_DIR / "diagnostico_stain.json", "w"), indent=2)

## 3. Descritores clássicos — o herdado e o novo (H3-b)

Quatro movimentos, em ordem de custo-benefício:

**(1) Transformada de Hellinger (√h).** Histogramas LBP vivem no simplex de probabilidade; aplicar geometria euclidiana (`StandardScaler` + LogReg) sobre eles é o erro geométrico padrão. `h → √h` antes de padronizar aproxima a distância euclidiana da distância de Hellinger. Custo: uma linha, zero extração nova. É a maior relação custo-benefício do projeto inteiro — e é aplicável **retroativamente** ao cache LBP de 54-d já existente.

**(2) LBP-HF (Ahonen et al., 2009).** O `riu2` obtém invariância rotacional **destruindo a orientação local** — ele é estruturalmente cego a textura anisotrópica, que é exatamente o regime das classes onde H4 prevê o clássico mais competitivo (STR, MUS). O LBP-HF obtém invariância rotacional pela **magnitude da DFT sobre o grupo de rotações**, preservando as relações de orientação. Kather et al. (2016) usaram LBP-HF; usar `riu2` sem justificativa é um flanco aberto na seção de Métodos. Aqui o notebook **mede** a diferença em vez de argumentar, inclusive na massa fora da diagonal STR↔MUS.

**(3) Cor (H3-b).** Aviso não-óbvio: **LBP por canal é invariante a ganho monotônico por canal**, portanto qualquer "constância de cor" global (gray-world, shades-of-gray) aplicada antes é **literalmente um no-op** sobre os códigos LBP. Adicionar cor de verdade exige informação *cruzada entre canais* — daí o **opponent-color LBP** (centro no canal A, vizinhos no canal B). O braço de cor é: LBP por canal RGB (162-d) + opponent nos 3 pares (84-d).

**(4) Fractal volumétrico — o risco de venue do §4-b.** O Clássico-B v1 (DBC + lacunaridade + slope de Fourier) é canônico mas é a **família básica**; o comitê UNICAMP/UFSCar cultiva **Bouligand–Minkowski** e **multifractal**. Aqui entra `Fractal-BM`: dilatação volumétrica da superfície de intensidade em 3D via `distance_transform_edt` (curva log V(r), 16-d) + **espectro multifractal** por momentos-q da medida de intensidade (9-d) + lacunaridade *gliding-box* (10-d) = **35-d**. Isso converte "espantalho da minha própria família" em "comparei a família básica e a volumétrica/multifractal, e reporto as duas".

In [ ]:
# ============================================================================
# 3. Implementacao dos descritores
# ============================================================================
LBP_PR  = [(8, 1), (16, 2), (24, 3)]     # herdado (Classico-A)
HF_PR   = [(8, 1), (16, 2)]              # LBP-HF: P=24 explode a LUT (2^24) -> omitido, declarar
OPP_PR  = [(8, 1), (16, 2)]
OPP_PAIRS = [(0, 1), (1, 2), (2, 0)]     # (R,G) (G,B) (B,R)

# ---------- LBP riu2 (reproducao exata do gate/Semana 2) ----------
def lbp_riu2_hist(g, P, R):
    codes = local_binary_pattern(g, P, R, method="uniform")
    h, _ = np.histogram(codes.ravel(), bins=np.arange(P + 3), density=True)
    return h.astype(np.float32)

def desc_lbp54(gray):
    return np.concatenate([lbp_riu2_hist(gray, P, R) for P, R in LBP_PR])

# ---------- LBP-HF (Ahonen 2009): magnitude da DFT sobre o grupo de rotacoes ----------
_HF_LUT, _RIU2_LUT, _GRID = {}, {}, {}

def hf_lut(P):
    if P in _HF_LUT: return _HF_LUT[P]
    n = 1 << P
    codes = np.arange(n, dtype=np.uint32)
    bits = ((codes[:, None] >> np.arange(P, dtype=np.uint32)) & 1).astype(np.uint8)
    circ = np.concatenate([bits, bits[:, :1]], axis=1).astype(np.int16)
    trans = np.abs(np.diff(circ, axis=1)).sum(1)
    ones = bits.sum(1)
    rot = ((bits == 1) & (np.roll(bits, 1, axis=1) == 0)).argmax(1)
    G = (P - 1) * P
    lut = np.full(n, G + 2, dtype=np.int32)          # nao-uniforme
    unif = trans <= 2
    m = unif & (ones > 0) & (ones < P)
    lut[m] = (ones[m] - 1) * P + rot[m]
    lut[unif & (ones == 0)] = G
    lut[unif & (ones == P)] = G + 1
    _HF_LUT[P] = lut
    return lut

def lbp_hf_block(g, P, R):
    codes = local_binary_pattern(g, P, R, method="default").astype(np.int64)
    cnt = np.bincount(hf_lut(P)[codes.ravel()], minlength=(P - 1) * P + 3).astype(np.float64)
    cnt /= max(cnt.sum(), 1e-12)
    H = cnt[:(P - 1) * P].reshape(P - 1, P)
    F = np.abs(np.fft.rfft(H, axis=1))[:, :P // 2 + 1]     # invariante a rotacao, preserva orientacao relativa
    return np.concatenate([F.ravel(), cnt[(P - 1) * P:]]).astype(np.float32)

def desc_lbphf(gray):
    return np.concatenate([lbp_hf_block(gray, P, R) for P, R in HF_PR])

# ---------- Opponent-color LBP (Maenpaa & Pietikainen) ----------
def riu2_lut(P):
    if P in _RIU2_LUT: return _RIU2_LUT[P]
    n = 1 << P
    codes = np.arange(n, dtype=np.uint32)
    bits = ((codes[:, None] >> np.arange(P, dtype=np.uint32)) & 1).astype(np.uint8)
    circ = np.concatenate([bits, bits[:, :1]], axis=1).astype(np.int16)
    trans = np.abs(np.diff(circ, axis=1)).sum(1)
    _RIU2_LUT[P] = np.where(trans <= 2, bits.sum(1), P + 1).astype(np.int32)
    return _RIU2_LUT[P]

def _grid(shape):
    if shape not in _GRID:
        _GRID[shape] = np.mgrid[0:shape[0], 0:shape[1]].astype(np.float32)
    return _GRID[shape]

def sample_ring(img, P, R):
    yy, xx = _grid(img.shape)
    f = img.astype(np.float32)
    out = np.empty((P,) + img.shape, dtype=np.float32)
    for p in range(P):
        ang = 2.0 * np.pi * p / P
        out[p] = map_coordinates(f, [yy - R * np.sin(ang), xx + R * np.cos(ang)],
                                 order=1, mode="nearest")
    return out

def opponent_hist(chC, chN, P, R):
    nb = sample_ring(chN, P, R)
    ctr = chC.astype(np.float32)
    codes = np.zeros(ctr.shape, dtype=np.int64)
    for p in range(P):
        codes |= ((nb[p] >= ctr).astype(np.int64) << p)
    h = np.bincount(riu2_lut(P)[codes.ravel()], minlength=P + 2).astype(np.float32)
    return h / max(h.sum(), 1e-12)

def desc_lbpcolor(rgb):
    per = [lbp_riu2_hist(rgb[:, :, c], P, R) for c in range(3) for P, R in LBP_PR]   # 162-d
    if not USE_OPPONENT_COLOR:
        return np.concatenate(per)
    opp = [opponent_hist(rgb[:, :, a], rgb[:, :, b], P, R)
           for (a, b) in OPP_PAIRS for P, R in OPP_PR]                                # 84-d
    return np.concatenate(per + opp)

# ---------- Bouligand-Minkowski (dilatacao volumetrica 3D) ----------
BM_RADII = np.array([1, 2, 3, 4, 5, 6, 7, 8, 10, 12, 14, 16, 20, 24, 28], dtype=np.float32)

def desc_bm(gray):
    g = np.asarray(Image.fromarray(gray).resize((BM_S, BM_S), Image.BILINEAR))
    z = (g.astype(np.int32) * (BM_Z - 1)) // 255
    vol = np.zeros((BM_S, BM_S, BM_Z), dtype=bool)
    xx, yy = np.meshgrid(np.arange(BM_S), np.arange(BM_S), indexing="ij")
    vol[xx.ravel(), yy.ravel(), z.ravel()] = True
    d = distance_transform_edt(~vol)
    V = np.array([float((d <= r).sum()) for r in BM_RADII])
    logV = np.log(np.maximum(V, 1.0)) / np.log(float(vol.size))       # curva normalizada, 15-d
    m = BM_RADII <= 10
    slope = np.polyfit(np.log(BM_RADII[m]), np.log(np.maximum(V[m], 1.0)), 1)[0]
    return np.concatenate([logV, [3.0 - slope]]).astype(np.float32)   # 16-d

# ---------- Espectro multifractal (momentos-q da medida de intensidade) ----------
MF_BOXES = [2, 4, 7, 8, 14, 16, 28, 32, 56]
MF_Q = np.arange(-5.0, 5.01, 1.0)

def desc_mf(gray):
    Nn = gray.shape[0]
    I = gray.astype(np.float64) + 1.0
    tot = I.sum()
    logr, logZ = [], []
    for r in MF_BOXES:
        nb = Nn // r
        mu = I[:nb * r, :nb * r].reshape(nb, r, nb, r).sum(axis=(1, 3)) / tot
        mu = np.maximum(mu.ravel(), 1e-12)
        logZ.append(np.log((mu[None, :] ** MF_Q[:, None]).sum(axis=1)))
        logr.append(np.log(r / float(Nn)))
    tau = np.polyfit(np.array(logr), np.array(logZ), 1)[0]
    alpha = np.gradient(tau, MF_Q)
    f = MF_Q * alpha - tau
    a_min, a_max = alpha.min(), alpha.max()
    i0 = int(np.argmax(f)); a0 = alpha[i0]
    width = a_max - a_min
    asym = (a0 - a_min) / max(width, 1e-9)
    qi = {q: int(np.argmin(np.abs(MF_Q - q))) for q in (-5, 0, 5)}
    return np.array([a_min, a_max, width, a0, f[i0], asym,
                     tau[qi[-5]], tau[qi[0]], tau[qi[5]]], dtype=np.float32)   # 9-d

# ---------- Lacunaridade gliding-box ----------
LAC_R = [2, 3, 4, 6, 8, 12, 16, 24, 32, 48]

def desc_lac(gray):
    I = gray.astype(np.float64)
    out = []
    for r in LAC_R:
        m = uniform_filter(I, size=r, mode="reflect")
        s = uniform_filter(I * I, size=r, mode="reflect")
        var = np.maximum(s - m * m, 0.0)
        mm = m.mean()
        out.append(np.log1p(var.mean() / max(mm * mm, 1e-9) + 1.0))
    return np.array(out, dtype=np.float32)   # 10-d

def desc_fractal_bm(gray):
    return np.concatenate([desc_bm(gray), desc_mf(gray), desc_lac(gray)])   # 35-d

# ---------- transformada de Hellinger / power-normalization ----------
def hellinger(X):
    return np.sqrt(np.clip(np.asarray(X, dtype=np.float32), 0.0, None))

print("descritores definidos")

In [ ]:
# ============================================================================
# 3.1 Self-tests dos descritores (rodar SEMPRE antes de extrair 100k)
# ============================================================================
rng = np.random.default_rng(0)
x = np.arange(224)
aniso = np.tile((127 + 100 * np.sin(2 * np.pi * x / 8.0)).astype(np.uint8), (224, 1))  # listras
aniso90 = np.rot90(aniso).copy()
iso = rng.integers(0, 255, size=(224, 224), dtype=np.uint8)
rgb_t = np.stack([iso, np.roll(iso, 3, 0), np.roll(iso, 5, 1)], -1)

t0 = time.time(); v_lbp = desc_lbp54(aniso);      t_lbp = time.time() - t0
t0 = time.time(); v_hf  = desc_lbphf(aniso);      t_hf  = time.time() - t0
t0 = time.time(); v_col = desc_lbpcolor(rgb_t);   t_col = time.time() - t0
t0 = time.time(); v_bm  = desc_fractal_bm(aniso); t_bm  = time.time() - t0

for nome, v, t in [("LBP54", v_lbp, t_lbp), ("LBP-HF", v_hf, t_hf),
                   ("LBP-cor", v_col, t_col), ("Fractal-BM", v_bm, t_bm)]:
    assert np.isfinite(v).all(), f"{nome} produziu NaN/Inf"
    print(f"{nome:11s} dim={v.shape[0]:4d}  {t*1000:7.1f} ms/patch  "
          f"ETA 100k @{N_WORKERS}w = {t*N/N_WORKERS/60:5.1f} min")

# (i) invariancia rotacional preservada nos dois; (ii) HF separa anisotropico de isotropico MAIS
d_rot_riu2 = np.linalg.norm(hellinger(desc_lbp54(aniso)) - hellinger(desc_lbp54(aniso90)))
d_rot_hf   = np.linalg.norm(hellinger(desc_lbphf(aniso)) - hellinger(desc_lbphf(aniso90)))
d_tex_riu2 = np.linalg.norm(hellinger(desc_lbp54(aniso)) - hellinger(desc_lbp54(iso)))
d_tex_hf   = np.linalg.norm(hellinger(desc_lbphf(aniso)) - hellinger(desc_lbphf(iso)))
print(f"\nriu2 : dist(rot90)={d_rot_riu2:.4f}  dist(aniso,iso)={d_tex_riu2:.4f}")
print(f"HF   : dist(rot90)={d_rot_hf:.4f}  dist(aniso,iso)={d_tex_hf:.4f}")
assert d_rot_hf < 1e-3, "LBP-HF perdeu invariancia rotacional - revisar LUT"
assert d_tex_hf > d_tex_riu2, ("LBP-HF nao esta separando anisotropico melhor que riu2 - "
                               "se isso ocorrer com dados reais e ACHADO, mas no sintetico e bug")
print("\nself-tests OK")

In [ ]:
# ============================================================================
# 4. Infra de extracao paralela + cache
# ============================================================================
DESC_SPEC = {
    "LBP54":      dict(fn=desc_lbp54,      mode="L",   version="v1"),
    "LBPHF":      dict(fn=desc_lbphf,      mode="L",   version="v1"),
    "LBPCOR":     dict(fn=desc_lbpcolor,   mode="RGB", version=("v1o" if USE_OPPONENT_COLOR else "v1p")),
    "FRACTALBM":  dict(fn=desc_fractal_bm, mode="L",   version="v1"),
}

def _load(p, mode):
    return np.asarray(Image.open(p).convert(mode))

def _chunk_worker(args):
    name, plist = args
    spec = DESC_SPEC[name]
    fn, mode = spec["fn"], spec["mode"]
    return np.asarray([fn(_load(p, mode)) for p in plist], dtype=np.float32)

def extract_desc(name, paths, tag, workers=None):
    spec = DESC_SPEC[name]
    fp = CACHE_DIR / f"{name}_{tag}_n{len(paths)}_{spec['version']}.npy"
    if fp.exists():
        print(f"[cache] {name}/{tag}: {fp.name}")
        return np.load(fp, mmap_mode="r")
    workers = workers or N_WORKERS
    chunks = [paths[i:i + CHUNK] for i in range(0, len(paths), CHUNK)]
    t0 = time.time(); parts = []
    with ProcessPoolExecutor(workers) as ex:
        for k, arr in enumerate(ex.map(_chunk_worker, [(name, c) for c in chunks])):
            parts.append(arr)
            if (k + 1) % 25 == 0 or k + 1 == len(chunks):
                done = sum(len(p) for p in parts); el = time.time() - t0
                print(f"  [{name}/{tag}] {done}/{len(paths)}  {el/60:.1f} min  "
                      f"ETA {(el/max(done,1))*(len(paths)-done)/60:.1f} min", flush=True)
    X = np.concatenate(parts).astype(np.float32)
    assert X.shape[0] == len(paths)
    np.save(fp, X)
    print(f"[extract] {name}/{tag}: {X.shape} -> {fp.name}  ({(time.time()-t0)/60:.1f} min)")
    return np.load(fp, mmap_mode="r")

print("infra de extracao pronta")

## 5. Caches herdados: carregar **e verificar alinhamento**

Este é o ponto de falha silenciosa mais perigoso do projeto. As features das Semanas 2 e 3-4 estão alinhadas à ordem canônica de `paths_100k`; se a ordenação do sistema de arquivos, a versão da lib ou o conteúdo da pasta mudaram desde então, o `.npy` continua carregando **e a acurácia despenca sem erro**. Duas verificações, nesta ordem:

1. **Reprodução bit-a-bit do LBP** em uma amostra: o descritor de 54-d é reimplementado aqui de forma idêntica, então linhas cacheadas devem bater com linhas recomputadas.
2. **Regressão contra os números registrados** (`frac=5%`, interno): LBP 80,39 · Fractal 64,04 · Phikon 99,37 · SimCLR ~95,2-95,4. Desvio maior que ±1,5 pt é desalinhamento até prova em contrário — **não** prossiga para o toque único no 7k com essa flag acesa.

In [ ]:
# ============================================================================
# 5. Carregar caches herdados
# ============================================================================
def _glob_one(d: Path, stem: str, n_expected: int):
    if not d.exists(): return None
    hits = sorted(d.glob(f"{stem}_n{n_expected}_*.npy")) or sorted(d.glob(f"{stem}_n*.npy"))
    for h in hits:
        X = np.load(h, mmap_mode="r")
        if X.shape[0] == n_expected:
            print(f"  [ok] {stem}: {h.name} {X.shape}")
            return X
        print(f"  [skip] {h.name}: n={X.shape[0]} != {n_expected}")
    return None

print("Semana 2 (100k):")
X_lbp_w2  = _glob_one(WEEK2_CACHE_DIR, "lbp_100k", N)
X_frac_w2 = _glob_one(WEEK2_CACHE_DIR, "frac_100k", N)
X_phik_w2 = _glob_one(WEEK2_CACHE_DIR, "phikon_100k", N)

print("Semanas 3-4 (100k):")
X_ssl = {}
for sd in SIMCLR_SEEDS:
    Xs = _glob_one(WEEK34_CACHE_DIR, f"simclr_s{sd}_{SIMCLR_AUG}_100k", N)
    if Xs is not None: X_ssl[sd] = Xs
X_ssl_abl = {}
for sd in SIMCLR_SEEDS:
    Xs = _glob_one(WEEK34_CACHE_DIR, f"simclr_s{sd}_{SIMCLR_AUG_ABL}_100k", N)
    if Xs is not None: X_ssl_abl[sd] = Xs

print("\nSSL full disponiveis:", sorted(X_ssl), "| SSL ablado disponiveis:", sorted(X_ssl_abl))
if not X_ssl:
    print("[ALERTA] nenhuma feature SimCLR encontrada -> o eixo SSL sai da tabela do 7k e o paper")
    print("         recua para o §10-3 (FM-ref congelado vs. classico). REGISTRAR.")

In [ ]:
# ============================================================================
# 5.1 Verificacao 1: reproducao bit-a-bit do LBP cacheado
# ============================================================================
if X_lbp_w2 is not None:
    rs = np.random.default_rng(7).choice(N, size=64, replace=False)
    ref = np.asarray([desc_lbp54(_load(paths_100k[i], "L")) for i in rs])
    got = np.asarray(X_lbp_w2[rs])
    err = np.abs(ref - got).max()
    ok = err < 1e-4
    print(f"LBP cache vs recomputo: erro max = {err:.2e} -> {'ALINHADO' if ok else 'DESALINHADO'}")
    if not ok:
        print("  Causas provaveis: ordem de arquivos diferente, NCT_DIR trocado, ou versao de skimage.")
        print("  NAO prosseguir para o 7k. Re-extrair LBP com extract_desc('LBP54', paths_100k, '100k').")
    CACHE_LBP_OK = bool(ok)
else:
    CACHE_LBP_OK = False
    print("Sem cache LBP da Semana 2 -> sera re-extraido nesta fase (custo ~10-20 min).")

In [ ]:
# ============================================================================
# 6. Aparato de probe (identico ao das Semanas 2 e 3-4, sem vazamento)
# ============================================================================
def make_probe(pca_dim=None):
    steps = [("sc", StandardScaler())]
    if pca_dim is not None:
        steps += [("pca", PCA(n_components=pca_dim, random_state=0)), ("sc2", StandardScaler())]
    steps += [("clf", LogisticRegression(max_iter=PROBE_MAX_ITER, class_weight="balanced",
                                         n_jobs=PROBE_N_JOBS))]
    return Pipeline(steps)

def probe_fit_predict(Xfit_all, Xeval_all, idx_eval, frac, seed, pca_dim, transform=None):
    # Xfit_all indexado por idx_100k; Xeval_all pode ser o mesmo (interno) ou o 7k (externo)
    tr = idx_train_pool[label_subsample(frac, seed)]
    Xtr = np.asarray(Xfit_all[tr])
    Xte = np.asarray(Xeval_all[idx_eval]) if idx_eval is not None else np.asarray(Xeval_all)
    if transform is not None:
        Xtr, Xte = transform(Xtr), transform(Xte)
    pipe = make_probe(pca_dim)
    pipe.fit(Xtr, y_100k[tr])
    yp = pipe.predict(Xte)
    evr = (float(pipe.named_steps["pca"].explained_variance_ratio_.sum())
           if pca_dim is not None else None)
    return yp, len(tr), evr

def bal_acc(y_true, y_pred):
    return 100.0 * balanced_accuracy_score(y_true, y_pred)

def curva_interna(Xall, transform=None, pca_dim=None, fracoes=None, seeds=None):
    fracoes = fracoes or LABEL_FRACTIONS
    rows = []
    for frac in fracoes:
        ss = (seeds or LABEL_SEEDS) if frac < 1.0 else [(seeds or LABEL_SEEDS)[0]]
        vals = []
        for s in ss:
            yp, n_tr, _ = probe_fit_predict(Xall, Xall, idx_eval_int, frac, s, pca_dim, transform)
            vals.append(bal_acc(y_100k[idx_eval_int], yp))
        m = float(np.mean(vals))
        ci = 1.96 * float(np.std(vals, ddof=1)) / np.sqrt(len(vals)) if len(vals) > 1 else None
        rows.append(dict(frac=frac, bal_acc=round(m, 2),
                         ci95=(round(ci, 2) if ci is not None else None), n_seeds=len(vals)))
    return pd.DataFrame(rows)

print("probe pronto")

In [ ]:
# ============================================================================
# 5.2 Verificacao 2: regressao contra os numeros REGISTRADOS (frac=5%, interno)
# ============================================================================
ALVOS_REGISTRADOS_5PCT = {"LBP": 80.39, "Fractal": 64.04, "Phikon": 99.37, "SimCLR_s0": 95.24}
TOL = 1.5

def regressao_5pct(nome, X):
    if X is None: return None
    vals = []
    for s in LABEL_SEEDS[:3]:
        yp, _, _ = probe_fit_predict(X, X, idx_eval_int, 0.05, s, None)
        vals.append(bal_acc(y_100k[idx_eval_int], yp))
    m = float(np.mean(vals)); alvo = ALVOS_REGISTRADOS_5PCT.get(nome)
    d = None if alvo is None else m - alvo
    flag = "OK" if (d is None or abs(d) <= TOL) else "DIVERGENTE"
    print(f"  {nome:10s} 5% = {m:6.2f}  alvo registrado = {alvo}  delta = "
          f"{'n/a' if d is None else format(d, '+.2f')}  -> {flag}")
    return m

print("Regressao de alinhamento (3 seeds de probe, frac=5%, interno):")
REG = {}
for nome, X in [("LBP", X_lbp_w2), ("Fractal", X_frac_w2), ("Phikon", X_phik_w2),
                ("SimCLR_s0", X_ssl.get(HEADLINE_SSL_SEED))]:
    REG[nome] = regressao_5pct(nome, X)
CACHE_REGRESSAO_OK = all(
    (v is None) or (ALVOS_REGISTRADOS_5PCT.get(k) is None) or abs(v - ALVOS_REGISTRADOS_5PCT[k]) <= TOL
    for k, v in REG.items())
print(f"\nAlinhamento global: {'OK - liberado para o 7k' if CACHE_REGRESSAO_OK else 'FALHOU - NAO tocar o 7k'}")

## 7. Extração dos descritores novos no 100k

Custo dominante da fase (CPU, paralelo às demais tarefas). O braço de cor com *opponent* é o mais caro; se o relógio apertar, `USE_OPPONENT_COLOR=False` reduz `LBPCOR` a 162-d por-canal e corta ~60% do custo — mas então **a única fonte de informação cromática cruzada some**, e H3-b vira "LBP por canal", que é fraco como enriquecimento de cor. Preferir cortar o `LBPHF` antes disso (ele responde a Métodos, não a H3-b).

In [ ]:
# ============================================================================
# 7. Extracao no 100k (CPU)
# ============================================================================
if not CACHE_LBP_OK:
    X_lbp_w2 = extract_desc("LBP54", paths_100k, "100k")

X_lbphf_100k  = extract_desc("LBPHF",     paths_100k, "100k")
X_lbpcor_100k = extract_desc("LBPCOR",    paths_100k, "100k")
X_fracbm_100k = extract_desc("FRACTALBM", paths_100k, "100k")
print("dims:", X_lbp_w2.shape[1], X_lbphf_100k.shape[1], X_lbpcor_100k.shape[1], X_fracbm_100k.shape[1])

## 8. H3-b — o lado clássico enriquecido (medido, não argumentado)

Escada de enriquecimento, cada degrau isolando **uma** mudança. Tudo no split interno (diagnóstico); a versão do paper sai do 7k.

| Braço | O que muda | Pergunta que responde |
|---|---|---|
| `LBP` | herdado (riu2, 54-d, euclidiano) | linha de base do gate |
| `LBP-H` | + Hellinger | a geometria estava errada? |
| `LBP-HF-H` | + orientação preservada | quanto o `riu2` custou? |
| `LBP-COR-H` | + cor (canal + opponent) | H3-b, eixo cor |
| `LBP-ENR` | HF + cor concatenados | melhor clássico LBP possível neste orçamento |
| `Fractal` | herdado (DBC + lac + Fourier) | família fractal básica |
| `Fractal-BM` | volumétrico + multifractal | a resposta ao §4-b (risco de venue) |
| `CLASSICO-MAX` | LBP-ENR + Fractal-BM | o clássico mais forte que este trabalho consegue |

**Guarda de leitura `[obrigatória]`:** o degrau escolhido para o paper deve ser escolhido **agora**, no interno, e congelado antes do toque no 7k. Escolher depois de ver o 7k é *cherry-picking* de baseline — exatamente a crítica que o enquadramento "clássico é cidadão de primeira classe" existe para evitar.

In [ ]:
# ============================================================================
# 8. H3-b: escada de enriquecimento classico (interno, diagnostico)
# ============================================================================
def concat_arms(*Xs):
    # concatena colunas materializando so o necessario (evita copia de 100k x N em memoria)
    return np.concatenate([np.asarray(X, dtype=np.float32) for X in Xs], axis=1)

X_lbp_enr    = concat_arms(hellinger(X_lbphf_100k), hellinger(X_lbpcor_100k))
X_class_max  = concat_arms(X_lbp_enr, X_fracbm_100k)

ARMS_INT = {
    "LBP":          (X_lbp_w2, None),
    "LBP-H":        (X_lbp_w2, hellinger),
    "LBP-HF-H":     (X_lbphf_100k, hellinger),
    "LBP-COR-H":    (X_lbpcor_100k, hellinger),
    "LBP-ENR":      (X_lbp_enr, None),
    "Fractal":      (X_frac_w2, None),
    "Fractal-BM":   (X_fracbm_100k, None),
    "CLASSICO-MAX": (X_class_max, None),
}
ARMS_INT = {k: v for k, v in ARMS_INT.items() if v[0] is not None}

FRACS_H3B = [f for f in [0.01, 0.05, 1.00] if f in LABEL_FRACTIONS] or list(LABEL_FRACTIONS)
rows = []
for nome, (X, tf) in ARMS_INT.items():
    c = curva_interna(X, transform=tf, fracoes=FRACS_H3B, seeds=LABEL_SEEDS[:3])
    c["arm"] = nome; c["dim"] = X.shape[1]
    rows.append(c)
h3b = pd.concat(rows, ignore_index=True)
h3b_piv = h3b.pivot_table(index=["arm", "dim"], columns="frac", values="bal_acc")
h3b.to_csv(OUT_DIR / "h3b_escada_classica_interno.csv", index=False)
print("=== H3-b: escada classica (bal.acc interna, 3 seeds) ===")
print(h3b_piv.round(2).to_string())

base = h3b_piv.loc[[i for i in h3b_piv.index if i[0] == "LBP"][0]]
print("\nGanho sobre o LBP herdado (pts):")
for idx in h3b_piv.index:
    if idx[0] == "LBP": continue
    print(f"  {idx[0]:14s} d={idx[1]:4d} " +
          "  ".join(f"{int(f*100):3d}%: {h3b_piv.loc[idx, f]-base[f]:+6.2f}" for f in FRACS_H3B))

In [ ]:
# ============================================================================
# 8.1 DECISAO TRAVADA: qual classico vai para o paper (escolher AGORA, antes do 7k)
# ============================================================================
# Regra pre-declarada: o classico-manchete e o de maior bal.acc media nas 3 fracoes;
# empate dentro de 0,5 pt resolve pelo MENOR numero de dimensoes (parcimonia).
med = h3b_piv.mean(axis=1)
best = med.sort_values(ascending=False)
cand = [i for i in best.index if best[i] >= best.iloc[0] - 0.5]
CLASSICO_MANCHETE = sorted(cand, key=lambda i: i[1])[0][0]
print("Ranking (media das 3 fracoes):")
print(best.round(2).to_string())
print(f"\n>>> CLASSICO-MANCHETE TRAVADO = {CLASSICO_MANCHETE}")
print(">>> Registrar com data no .md. Trocar isso depois de ver o 7k e cherry-picking.")

## 9. H4 no split interno — inclusive o teste direto da cegueira do `riu2`

Duas coisas aqui, e a segunda é a que vale mais para a discussão:

- **H4 pré-registrada:** o gap aprendido−clássico é maior nas morfológicas {TUM, NORM} do que nas texturais {STR, MUS, ADI, MUC}.
- **Mecanismo:** se a explicação for "o `riu2` é cego a anisotropia", então a massa fora da diagonal **STR↔MUS** deve cair quando se troca `riu2` por `HF`. Se não cair, a explicação está errada e o texto não pode usá-la. Este é um teste que ou dá munição ou tira uma frase falsa do paper.

In [ ]:
# ============================================================================
# 9. H4 interno + confusao STR<->MUS
# ============================================================================
def preds_internas(X, tf=None, frac=1.0, seed=0, pca_dim=None):
    yp, _, _ = probe_fit_predict(X, X, idx_eval_int, frac, seed, pca_dim, tf)
    return yp

y_int = y_100k[idx_eval_int]

def recall_por_classe(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=range(len(CLASSES)))
    return 100.0 * np.diag(cm) / np.maximum(cm.sum(1), 1), cm

H4_ARMS_INT = {k: ARMS_INT[k] for k in ["LBP", CLASSICO_MANCHETE] if k in ARMS_INT}
if X_phik_w2 is not None: H4_ARMS_INT["Phikon"] = (X_phik_w2, None)
if HEADLINE_SSL_SEED in X_ssl: H4_ARMS_INT["SimCLR"] = (X_ssl[HEADLINE_SSL_SEED], None)

rec_int, cms_int = {}, {}
for nome, (X, tf) in H4_ARMS_INT.items():
    yp = preds_internas(X, tf, frac=1.0, seed=LABEL_SEEDS[0])
    r, cm = recall_por_classe(y_int, yp)
    rec_int[nome] = r; cms_int[nome] = cm
    print(f"{nome:14s} bal.acc = {r.mean():6.2f}")

tab = pd.DataFrame(rec_int, index=CLASSES).round(2)
tab["tipo_H4"] = ["morfologica" if c in H4_MORFO else
                  ("textural" if c in H4_TEXTURAL else "nao-preregistrada") for c in CLASSES]
print("\n=== Recall por classe (interno, frac=100%) ===")
print(tab.to_string())
tab.to_csv(OUT_DIR / "h4_recall_interno.csv")

def massa_fora_diagonal(cm, a, b):
    ia, ib = CLASSES.index(a), CLASSES.index(b)
    tot = cm[ia].sum() + cm[ib].sum()
    return 100.0 * (cm[ia, ib] + cm[ib, ia]) / max(tot, 1)

print("\n=== Confusao STR<->MUS (teste da cegueira do riu2 a anisotropia) ===")
for nome in ["LBP", "LBP-HF-H", CLASSICO_MANCHETE]:
    if nome not in ARMS_INT: continue
    X, tf = ARMS_INT[nome]
    yp = preds_internas(X, tf, frac=1.0, seed=LABEL_SEEDS[0])
    _, cm = recall_por_classe(y_int, yp)
    print(f"  {nome:14s} massa STR<->MUS = {massa_fora_diagonal(cm, 'STR', 'MUS'):5.2f}%  "
          f"| ADI<->MUC = {massa_fora_diagonal(cm, 'ADI', 'MUC'):5.2f}%")
print("Se a massa STR<->MUS NAO cair de riu2 para HF, a explicacao 'riu2 e cego a orientacao'")
print("NAO se sustenta com estes dados - remover a frase do paper, nao suavizar.")

## 10. Pré-commit do toque único no 7k `[trava]`

A disciplina de toque único não é "rodar uma vez"; é **não iterar a especificação depois de ver o resultado**. A célula abaixo serializa a especificação completa (braços, condições, frações, seeds, classificador, transformações) e grava o hash. A execução do 7k só prossegue se o hash bater com o gravado — retomada após queda é permitida, **mudança de escopo não é**.

Nota de escopo: incluir **todos** os braços numa única execução não fere o toque único (o que fere é rodar, olhar, ajustar e rodar de novo). Incluir tudo agora é o que permite reportar a escada H3-b completa no 7k sem uma segunda visita.

In [ ]:
# ============================================================================
# 10. Pre-commit
# ============================================================================
ARMS_7K_SPEC = {}
ARMS_7K_SPEC["LBP"]        = dict(desc="LBP54",     transform=None,        origem="semana2")
ARMS_7K_SPEC["LBP-H"]      = dict(desc="LBP54",     transform="hellinger", origem="semana2")
ARMS_7K_SPEC["LBP-HF-H"]   = dict(desc="LBPHF",     transform="hellinger", origem="semana56")
ARMS_7K_SPEC["LBP-COR-H"]  = dict(desc="LBPCOR",    transform="hellinger", origem="semana56")
ARMS_7K_SPEC["LBP-ENR"]    = dict(desc="LBPHF+LBPCOR", transform="hellinger_pre", origem="semana56")
ARMS_7K_SPEC["Fractal"]    = dict(desc="FRACTALV1", transform=None,        origem="semana2")
ARMS_7K_SPEC["Fractal-BM"] = dict(desc="FRACTALBM", transform=None,        origem="semana56")
ARMS_7K_SPEC["CLASSICO-MAX"] = dict(desc="LBPHF+LBPCOR+FRACTALBM", transform="hellinger_pre",
                                    origem="semana56")
if X_phik_w2 is not None:
    ARMS_7K_SPEC["Phikon"] = dict(desc="PHIKON", transform=None, origem="gate")
for sd in sorted(X_ssl):
    ARMS_7K_SPEC[f"SimCLR_s{sd}"] = dict(desc=f"SIMCLR_s{sd}_{SIMCLR_AUG}", transform=None,
                                         origem="semanas34")
for sd in sorted(X_ssl_abl):
    ARMS_7K_SPEC[f"SimCLR_abl_s{sd}"] = dict(desc=f"SIMCLR_s{sd}_{SIMCLR_AUG_ABL}",
                                             transform=None, origem="semanas34")

PRECOMMIT = dict(
    data_precommit=time.strftime("%Y-%m-%d %H:%M"),
    arms=ARMS_7K_SPEC,
    classico_manchete=CLASSICO_MANCHETE,
    ssl_manchete=(f"SimCLR_s{HEADLINE_SSL_SEED}" if HEADLINE_SSL_SEED in X_ssl else None),
    condicoes=["RAW", f"PCA{COMMON_D}"],
    fracoes=LABEL_FRACTIONS, seeds=LABEL_SEEDS,
    probe="StandardScaler+LogisticRegression(max_iter=2000, class_weight=balanced)",
    pca="Pipeline(sc->PCA(d)->sc->logreg), fit so no fold de treino, refeito por (frac,seed)",
    treino="subamostra estratificada de idx_train_pool (80k) do NCT-100K",
    teste="CRC-VAL-HE-7K inteiro, ordem canonica",
    metricas=["balanced_accuracy", "f1_macro", "recall_por_classe", "matriz_confusao"],
    estatistica="diferenca pareada + IC bootstrap estratificado sobre o teste (B=%d) + Holm" % BOOTSTRAP_B,
)
spec_hash = hashlib.sha256(json.dumps(PRECOMMIT["arms"], sort_keys=True).encode()).hexdigest()[:16]
PRECOMMIT["spec_hash"] = spec_hash

if PRECOMMIT_PATH.exists():
    old = json.loads(PRECOMMIT_PATH.read_text())
    if old["spec_hash"] != spec_hash:
        raise RuntimeError(
            f"Pre-commit JA existe com hash {old['spec_hash']} != {spec_hash}.\n"
            "A especificacao do toque unico mudou. Isso quebra a disciplina de toque unico.\n"
            "Se a mudanca for legitima (ex.: um braco nao existia), apague o arquivo A MAO e "
            "REGISTRE no .md a data, o motivo e o que ja tinha sido visto do 7k.")
    print(f"Pre-commit conferido (hash {spec_hash}) - retomada permitida.")
else:
    PRECOMMIT_PATH.write_text(json.dumps(PRECOMMIT, indent=2))
    print(f"Pre-commit GRAVADO (hash {spec_hash}) em {PRECOMMIT_PATH}")
print("Bracos:", list(ARMS_7K_SPEC))

## 11. Features do 7k

Três famílias, três riscos diferentes:

- **Clássicas:** determinísticas, reimplementadas aqui — sem risco.
- **Phikon:** o pré-processamento precisa ser **idêntico** ao que gerou o cache de 100k, senão o probe treinado no 100k recebe features de outra distribuição e o número do 7k fica errado *para baixo*, silenciosamente. A célula valida por similaridade cosseno contra linhas cacheadas antes de aceitar.
- **SimCLR:** mesmo risco, mais o de o checkpoint não existir. Validação idêntica.

Se a validação falhar, **pare**: um Phikon mal pré-processado no 7k destrói a âncora publicada (0,935) que é a única validação externa do pipeline inteiro.

In [ ]:
# ============================================================================
# 11.1 Descritores classicos no 7k
# ============================================================================
X7_lbp    = extract_desc("LBP54",     paths_7k, "7k")
X7_lbphf  = extract_desc("LBPHF",     paths_7k, "7k")
X7_lbpcor = extract_desc("LBPCOR",    paths_7k, "7k")
X7_fracbm = extract_desc("FRACTALBM", paths_7k, "7k")
print("7k classicos ok:", X7_lbp.shape, X7_lbphf.shape, X7_lbpcor.shape, X7_fracbm.shape)

In [ ]:
# ============================================================================
# 11.2 Fractal v1 (Semana 2) no 7k - reimplementacao para o braco herdado
#      DBC (Sarkar-Chaudhuri) + lacunaridade gliding-box + slope espectral = 38-d
# ============================================================================
DBC_BOXES = [4, 8, 16, 28, 56]

def _dbc_dim(g, boxes=DBC_BOXES):
    Nn = g.shape[0]; I = g.astype(np.float64)
    xs, ys = [], []
    for s in boxes:
        nb = Nn // s
        blk = I[:nb*s, :nb*s].reshape(nb, s, nb, s)
        mx = blk.max(axis=(1, 3)); mn = blk.min(axis=(1, 3))
        h = max(s * 255.0 / Nn, 1e-9)
        nr = np.floor(mx / h) - np.ceil(mn / h) + 1
        xs.append(np.log(1.0 / s)); ys.append(np.log(max(nr.sum(), 1.0)))
    return float(np.polyfit(xs, ys, 1)[0])

def _fourier_slope(g):
    F = np.fft.fftshift(np.abs(np.fft.fft2(g.astype(np.float64))) ** 2)
    Nn = g.shape[0]; c = Nn // 2
    yy, xx = np.mgrid[0:Nn, 0:Nn]
    r = np.sqrt((yy - c) ** 2 + (xx - c) ** 2).astype(int)
    rmax = c
    prof = np.bincount(r.ravel(), F.ravel(), minlength=rmax + 1)[1:rmax]
    cnt = np.bincount(r.ravel(), minlength=rmax + 1)[1:rmax]
    prof = prof / np.maximum(cnt, 1)
    m = prof > 0
    a, b = np.polyfit(np.log(np.arange(1, rmax)[m]), np.log(prof[m]), 1)
    return float(a), float(b)

def desc_fractal_v1(gray):
    if gray.std() < 1e-6:
        return np.zeros(38, dtype=np.float32)
    lac = desc_lac(gray)                                   # 10
    g_glob = _dbc_dim(gray)                                # 1
    t2 = [_dbc_dim(t) for t in _tiles(gray, 2)]            # 4
    t4 = [_dbc_dim(t) for t in _tiles(gray, 4)]            # 16
    st = [np.mean(t4), np.std(t4), np.min(t4), np.max(t4), np.median(t4)]   # 5
    sl, ic = _fourier_slope(gray)                          # 2
    return np.concatenate([lac, [g_glob], t2, t4, st, [sl, ic]]).astype(np.float32)

def _tiles(g, k):
    n = g.shape[0] // k
    return [g[i*n:(i+1)*n, j*n:(j+1)*n] for i in range(k) for j in range(k)]

DESC_SPEC["FRACTALV1"] = dict(fn=desc_fractal_v1, mode="L", version="clB_frac_v1r")
assert desc_fractal_v1(np.asarray(Image.open(paths_7k[0]).convert("L"))).shape[0] == 38
X7_fracv1 = extract_desc("FRACTALV1", paths_7k, "7k")

# Se o cache da Semana 2 nao existe, o braco Fractal-v1 e reconstruido aqui (100k).
if X_frac_w2 is None:
    print("[Fractal v1] sem cache da Semana 2 -> extraindo no 100k com a reimplementacao.")
    X_frac_w2 = extract_desc("FRACTALV1", paths_100k, "100k")
    if "Fractal" in ARMS_INT: ARMS_INT["Fractal"] = (X_frac_w2, None)

# Validacao: a reimplementacao bate com o cache de 100k da Semana 2?
if X_frac_w2 is not None:
    rs = np.random.default_rng(11).choice(N, size=48, replace=False)
    ref = np.asarray([desc_fractal_v1(_load(paths_100k[i], "L")) for i in rs])
    got = np.asarray(X_frac_w2[rs])
    cs = float(np.mean([np.dot(a, b) / (np.linalg.norm(a)*np.linalg.norm(b) + 1e-12)
                        for a, b in zip(ref, got)]))
    print(f"Fractal v1 reimplementado vs cache Semana 2: cos medio = {cs:.4f}")
    if cs < 0.995:
        print("  [ATENCAO] reimplementacao NAO reproduz o cache (ordem das familias/parametros diferem).")
        print("  Opcao segura: usar a reimplementacao nos DOIS lados -> X_frac_w2 = extract_desc('FRACTALV1', paths_100k, '100k')")
        X_frac_w2 = extract_desc("FRACTALV1", paths_100k, "100k")
        ARMS_INT["Fractal"] = (X_frac_w2, None)

In [ ]:
# ============================================================================
# 11.3 Phikon no 7k (com validacao contra o cache de 100k)
# ============================================================================
X7_phikon = None
if X_phik_w2 is not None:
    fp = CACHE_DIR / f"phikon_7k_n{N7}.npy"
    import torch
    from transformers import ViTModel, AutoImageProcessor
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    proc = AutoImageProcessor.from_pretrained("owkin/phikon")
    mdl = ViTModel.from_pretrained("owkin/phikon", add_pooling_layer=False).to(dev).eval()

    @torch.no_grad()
    def phikon_feats(plist, batch=64):
        out = []
        for i in range(0, len(plist), batch):
            ims = [Image.open(p).convert("RGB") for p in plist[i:i+batch]]
            inp = proc(ims, return_tensors="pt").to(dev)
            out.append(mdl(**inp).last_hidden_state[:, 0, :].float().cpu().numpy())
            if (i // batch) % 20 == 0: print(f"    phikon {i}/{len(plist)}", flush=True)
        return np.concatenate(out).astype(np.float32)

    rs = np.random.default_rng(13).choice(N, size=32, replace=False)
    ref = phikon_feats(paths_100k[rs]); got = np.asarray(X_phik_w2[rs])
    cs = float(np.mean([np.dot(a, b)/(np.linalg.norm(a)*np.linalg.norm(b)+1e-12)
                        for a, b in zip(ref, got)]))
    print(f"Phikon: cos(recomputo, cache 100k) = {cs:.5f}")
    assert cs > 0.999, ("Pre-processamento do Phikon DIVERGE do que gerou o cache. "
                        "Nao prosseguir: o probe treinado no 100k receberia outra distribuicao no 7k.")
    X7_phikon = np.load(fp, mmap_mode="r") if fp.exists() else None
    if X7_phikon is None:
        X7_phikon = phikon_feats(paths_7k); np.save(fp, X7_phikon)
        X7_phikon = np.load(fp, mmap_mode="r")
    print("Phikon 7k:", X7_phikon.shape)
    del mdl; torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
# ============================================================================
# 11.4 SimCLR no 7k (a partir dos checkpoints das Semanas 3-4)
# ============================================================================
X7_ssl, X7_ssl_abl = {}, {}
if X_ssl or X_ssl_abl:
    import torch, torchvision
    import torch.nn as nn, torch.nn.functional as Fn
    from torchvision import transforms as T
    from torch.utils.data import Dataset, DataLoader
    dev = "cuda" if torch.cuda.is_available() else "cpu"

    NORM_MEAN, NORM_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
    eval_tf = T.Compose([T.Resize(224), T.CenterCrop(224), T.ToTensor(),
                         T.Normalize(NORM_MEAN, NORM_STD)])

    class SimCLRNet(nn.Module):
        def __init__(self, backbone_name, proj_dim):
            super().__init__()
            base = torchvision.models.__dict__[backbone_name](weights=None)
            self.feat_dim = base.fc.in_features
            self.stem = nn.Sequential(base.conv1, base.bn1, base.relu, base.maxpool)
            self.stages = nn.Sequential(base.layer1, base.layer2, base.layer3, base.layer4)
            self.avgpool = base.avgpool
            self.grad_ckpt = False
            self.proj = nn.Sequential(nn.Linear(self.feat_dim, self.feat_dim), nn.ReLU(inplace=True),
                                      nn.Linear(self.feat_dim, proj_dim))
        def encode(self, x):
            return torch.flatten(self.avgpool(self.stages(self.stem(x))), 1)

    class DS(Dataset):
        def __init__(self, plist): self.p = plist
        def __len__(self): return len(self.p)
        def __getitem__(self, i): return eval_tf(Image.open(self.p[i]).convert("RGB"))

    @torch.no_grad()
    def ssl_feats(model, plist, batch=128):
        dl = DataLoader(DS(plist), batch_size=batch, shuffle=False, num_workers=min(4, N_WORKERS), pin_memory=True)
        return np.concatenate([model.encode(xb.to(dev)).float().cpu().numpy() for xb in dl]).astype(np.float32)

    def load_ckpt(sd, aug):
        ck_p = CKPT_DIR / f"simclr_s{sd}_{aug}_best.pt"
        if not ck_p.exists(): return None
        ck = torch.load(ck_p, map_location=dev)
        m = SimCLRNet(ck.get("backbone", SIMCLR_BACKBONE), SIMCLR_PROJ_DIM).to(dev)
        m.load_state_dict(ck["state_dict"]); m.eval()
        return m

    def extrai_ssl_7k(sd, aug, cache100):
        m = load_ckpt(sd, aug)
        if m is None:
            print(f"  [falta] checkpoint s{sd}_{aug} ausente -> braco fora do 7k"); return None
        rs = np.random.default_rng(17).choice(N, size=32, replace=False)
        ref = ssl_feats(m, paths_100k[rs]); got = np.asarray(cache100[rs])
        cs = float(np.mean([np.dot(a, b)/(np.linalg.norm(a)*np.linalg.norm(b)+1e-12)
                            for a, b in zip(ref, got)]))
        print(f"  s{sd}_{aug}: cos(recomputo, cache 100k) = {cs:.5f}")
        if cs < 0.999:
            print("   [ATENCAO] checkpoint NAO corresponde ao cache de features. Nao usar este braco.")
            return None
        fp = CACHE_DIR / f"simclr_s{sd}_{aug}_7k_n{N7}.npy"
        if not fp.exists(): np.save(fp, ssl_feats(m, paths_7k))
        del m
        return np.load(fp, mmap_mode="r")

    for sd, X in sorted(X_ssl.items()):
        r = extrai_ssl_7k(sd, SIMCLR_AUG, X)
        if r is not None: X7_ssl[sd] = r
    for sd, X in sorted(X_ssl_abl.items()):
        r = extrai_ssl_7k(sd, SIMCLR_AUG_ABL, X)
        if r is not None: X7_ssl_abl[sd] = r
    if torch.cuda.is_available(): torch.cuda.empty_cache()
print("SimCLR 7k:", sorted(X7_ssl), "| ablado:", sorted(X7_ssl_abl))

In [ ]:
# ============================================================================
# 11.5 Montagem final dos pares (features 100k, features 7k) por braco
# ============================================================================
def _pair(X100, X7, tf=None):
    if X100 is None or X7 is None: return None
    return (X100, X7, tf)

X_lbpenr_7k    = concat_arms(hellinger(X7_lbphf), hellinger(X7_lbpcor))
X_classmax_7k  = concat_arms(X_lbpenr_7k, X7_fracbm)

PAIRS = {}
PAIRS["LBP"]          = _pair(X_lbp_w2, X7_lbp, None)
PAIRS["LBP-H"]        = _pair(X_lbp_w2, X7_lbp, hellinger)
PAIRS["LBP-HF-H"]     = _pair(X_lbphf_100k, X7_lbphf, hellinger)
PAIRS["LBP-COR-H"]    = _pair(X_lbpcor_100k, X7_lbpcor, hellinger)
PAIRS["LBP-ENR"]      = _pair(X_lbp_enr, X_lbpenr_7k, None)
PAIRS["Fractal"]      = _pair(X_frac_w2, X7_fracv1, None)
PAIRS["Fractal-BM"]   = _pair(X_fracbm_100k, X7_fracbm, None)
PAIRS["CLASSICO-MAX"] = _pair(X_class_max, X_classmax_7k, None)
PAIRS["Phikon"]       = _pair(X_phik_w2, X7_phikon, None)
for sd in sorted(X7_ssl):     PAIRS[f"SimCLR_s{sd}"]     = _pair(X_ssl[sd], X7_ssl[sd], None)
for sd in sorted(X7_ssl_abl): PAIRS[f"SimCLR_abl_s{sd}"] = _pair(X_ssl_abl[sd], X7_ssl_abl[sd], None)
PAIRS = {k: v for k, v in PAIRS.items() if v is not None}

faltando = [k for k in ARMS_7K_SPEC if k not in PAIRS]
print("Bracos prontos para o 7k:", list(PAIRS))
if faltando:
    print("[AVISO] bracos do pre-commit sem features:", faltando,
          "-> registrar por que sairam (nao e cherry-picking se for indisponibilidade tecnica,")
    print("   mas TEM que estar escrito no .md).")
for k, (A, B, _) in PAIRS.items():
    assert A.shape[1] == B.shape[1], f"{k}: dimensao 100k != 7k"
    assert B.shape[0] == N7, f"{k}: n do 7k errado"
print("dimensoes conferidas")

## 12. Toque único no 7k — **estes são os números do paper**

Uma execução, retomável por configuração (cada `(braço, condição, fração, seed)` grava predições em disco e é pulada se já existir). O que é gravado: predições completas para os 7.180 patches — o que permite bootstrap, matriz de confusão, recall por classe e teste pareado **sem refit**.

Custo: aproximadamente `n_braços × 2 × 21` ajustes de regressão logística. Os ajustes em `frac=100%` (80k amostras) dominam. Se o relógio apertar, corte `0.10` e `0.25` das frações **antes** de cortar braços — a curva de rótulo tolera 3 pontos, a tabela de braços não tolera buracos.

In [ ]:
# ============================================================================
# 12. TOQUE UNICO NO 7K
# ============================================================================
assert CACHE_REGRESSAO_OK, "Alinhamento de cache falhou (celula 5.2). Corrigir antes do toque unico."
CONDS = [None, COMMON_D]

def _pred_path(arm, cond, frac, seed):
    c = "RAW" if cond is None else f"PCA{cond}"
    return PRED_DIR / f"{arm}__{c}__f{int(frac*100)}__s{seed}.npy"

def run_7k():
    linhas = []
    total = sum(len(CONDS) * sum(1 for f in LABEL_FRACTIONS
                                 for _ in (LABEL_SEEDS if f < 1.0 else LABEL_SEEDS[:1]))
                for _ in PAIRS)
    k = 0; t0 = time.time()
    for arm, (X100, X7, tf) in PAIRS.items():
        for cond in CONDS:
            for frac in LABEL_FRACTIONS:
                for seed in (LABEL_SEEDS if frac < 1.0 else LABEL_SEEDS[:1]):
                    k += 1
                    n_tr_prev = len(label_subsample(frac, seed))
                    if cond is not None and min(n_tr_prev, X100.shape[1]) < cond:
                        print(f"  [skip] {arm} PCA{cond} f{frac}: d>{min(n_tr_prev, X100.shape[1])} "
                              f"(n_treino ou dim menor que d). Registrar a lacuna.")
                        continue
                    pp = _pred_path(arm, cond, frac, seed)
                    if pp.exists():
                        yp = np.load(pp); evr = None
                    else:
                        yp, n_tr, evr = probe_fit_predict(X100, X7, None, frac, seed, cond, tf)
                        np.save(pp, yp.astype(np.int8))
                    linhas.append(dict(arm=arm, cond=("RAW" if cond is None else f"PCA{cond}"),
                                       frac=frac, seed=seed,
                                       bal_acc=bal_acc(y_7k, yp),
                                       f1_macro=100*f1_score(y_7k, yp, average="macro"),
                                       evr=evr, dim=X100.shape[1]))
                    if k % 10 == 0 or k == total:
                        el = (time.time()-t0)/60
                        print(f"  [{k}/{total}] {arm} {linhas[-1]['cond']} f{frac} s{seed} "
                              f"= {linhas[-1]['bal_acc']:.2f}  ({el:.1f} min)", flush=True)
    return pd.DataFrame(linhas)

df7 = run_7k()
df7.to_csv(OUT_DIR / "resultados_7k_FINAL.csv", index=False)
print("\n=== 7K FINAL - balanced accuracy (media sobre seeds) ===")
print(df7.pivot_table(index=["arm", "cond"], columns="frac", values="bal_acc").round(2).to_string())

## 13. Âncora de sanidade e estatística pareada

**Âncora.** O único braço com número publicado continua sendo o Phikon: **0,935 de acurácia balanceada** na tarefa CRC do benchmark **eva** (kaiko.ai; Gatopoulos, Känzig, Moser, Otálora — MIDL 2024; **Tabela 3**, "Linear probing evaluation of FMs, averaged balanced accuracy"), com NCT-CRC-HE-100K para treino e CRC-VAL-HE-7K para validação, probe linear sobre [CLS] congelado. **Item de bibliografia fechado:** autor-ano e número da tabela confirmados; falta só conferir a entrada BibTeX final.

Cuidado ao comparar: aqui o Phikon em `frac=100%` treina no **train_pool de 80k**, não nos 100k inteiros do gate e do eva. Esperar algo **ligeiramente abaixo** de 93,02%. Diferença grande (>2 pts) indica problema, não coorte.

**Estatística.** O `Wilcoxon` do §7 é substituído (guarda-e): com 5 seeds ele não consegue p<0,05 por construção. O que entra:

- diferença pareada braço-a-braço na **mesma** subamostra de rótulo (pareamento por seed);
- **IC 95% por bootstrap estratificado sobre os 7.180 patches de teste** — funciona inclusive em `frac=100%`, onde não há seeds, e é o IC que um revisor de estatística aceita;
- tamanho de efeito (diferença em pontos + `d` de Cohen pareado quando há seeds);
- **Holm** sobre a família de comparações da tabela manchete.

In [ ]:
# ============================================================================
# 13.1 Ancora de sanidade
# ============================================================================
EVA_ALVO = 93.5
if "Phikon" in PAIRS:
    v = df7[(df7.arm == "Phikon") & (df7.cond == "RAW") & (df7.frac == 1.0)]["bal_acc"].iloc[0]
    print(f"Phikon 7k RAW frac=100% (treino=train_pool 80k): {v:.2f}%")
    print(f"  alvo publicado (eva/kaiko, Tabela 3, treino=100k): {EVA_ALVO:.1f}%   delta = {v-EVA_ALVO:+.2f} pt")
    print(f"  gate (treino=100k inteiro): 93,02%   delta vs gate = {v-93.02:+.2f} pt")
    print("  Esperado: levemente abaixo do gate (20% menos dados de treino). |delta| > 2 pt = investigar.")

In [ ]:
# ============================================================================
# 13.2 Estatistica pareada: bootstrap estratificado sobre o teste 7k
# ============================================================================
def _boot_idx(y, B, seed=0):
    rng = np.random.default_rng(seed)
    per = [np.where(y == c)[0] for c in range(len(CLASSES))]
    return [np.concatenate([rng.choice(p, size=len(p), replace=True) for p in per]) for _ in range(B)]

BOOT = _boot_idx(y_7k, BOOTSTRAP_B, seed=GLOBAL_SEED)

def _ba(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=range(len(CLASSES)))
    return 100.0 * float(np.mean(np.diag(cm) / np.maximum(cm.sum(1), 1)))

def load_pred(arm, cond, frac, seed):
    p = _pred_path(arm, None if cond == "RAW" else COMMON_D, frac, seed)
    return np.load(p)

def ic_bootstrap(arm, cond, frac, seed=0):
    yp = load_pred(arm, cond, frac, seed)
    vals = np.array([_ba(y_7k[b], yp[b]) for b in BOOT])
    return _ba(y_7k, yp), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def diff_bootstrap(armA, armB, cond, frac, seed=0):
    a, b = load_pred(armA, cond, frac, seed), load_pred(armB, cond, frac, seed)
    d = np.array([_ba(y_7k[i], a[i]) - _ba(y_7k[i], b[i]) for i in BOOT])
    ponto = _ba(y_7k, a) - _ba(y_7k, b)
    return ponto, float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5)), float((d <= 0).mean())

def holm(pvals):
    order = np.argsort(pvals); m = len(pvals); adj = np.empty(m); run = 0.0
    for k, i in enumerate(order):
        run = max(run, (m - k) * pvals[i]); adj[i] = min(run, 1.0)
    return adj

SSL_MANCHETE = f"SimCLR_s{HEADLINE_SSL_SEED}" if f"SimCLR_s{HEADLINE_SSL_SEED}" in PAIRS else None
FRACS_STAT = [f for f in [0.01, 0.05, 1.00] if f in LABEL_FRACTIONS] or list(LABEL_FRACTIONS)
COMPS = []
for frac in FRACS_STAT:
    if SSL_MANCHETE:
        COMPS.append((SSL_MANCHETE, CLASSICO_MANCHETE, "RAW", frac))
        COMPS.append(("Phikon", SSL_MANCHETE, "RAW", frac))
    COMPS.append(("Phikon", CLASSICO_MANCHETE, "RAW", frac))
    COMPS.append((CLASSICO_MANCHETE, "LBP", "RAW", frac))
    if SSL_MANCHETE:
        COMPS.append((SSL_MANCHETE, CLASSICO_MANCHETE, f"PCA{COMMON_D}", frac))
    COMPS.append(("Phikon", CLASSICO_MANCHETE, f"PCA{COMMON_D}", frac))

rows = []
for A, B, cond, frac in COMPS:
    if A not in PAIRS or B not in PAIRS: continue
    d, lo, hi, p1 = diff_bootstrap(A, B, cond, frac)
    rows.append(dict(A=A, B=B, cond=cond, frac=frac, diff=round(d, 2),
                     ic_lo=round(lo, 2), ic_hi=round(hi, 2), p_boot_unilateral=round(p1, 4)))
stat7 = pd.DataFrame(rows)
if len(stat7):
    stat7["p_holm"] = holm(stat7["p_boot_unilateral"].values).round(4)
    stat7["significativo"] = (stat7.ic_lo > 0) | (stat7.ic_hi < 0)
stat7.to_csv(OUT_DIR / "estatistica_pareada_7k.csv", index=False)
print("=== Diferencas pareadas no 7k (IC95% bootstrap estratificado sobre o teste) ===")
print(stat7.to_string(index=False))

# Suplemento: Wilcoxon unilateral entre seeds (so onde ha >1 seed). Piso de p com n=5 e 0,03125
# bilateral 0,0625 - por isso NAO e o teste principal (guarda-e).
sup = []
for A, B, cond, frac in COMPS:
    if A not in PAIRS or B not in PAIRS or frac >= 1.0: continue
    a = df7[(df7.arm==A)&(df7.cond==cond)&(df7.frac==frac)].sort_values("seed")["bal_acc"].values
    b = df7[(df7.arm==B)&(df7.cond==cond)&(df7.frac==frac)].sort_values("seed")["bal_acc"].values
    if len(a) != len(b) or len(a) < 3: continue
    try:
        w = stats.wilcoxon(a, b, alternative="greater")
        d = (a - b).mean() / max((a - b).std(ddof=1), 1e-9)     # d de Cohen pareado
        sup.append(dict(A=A, B=B, cond=cond, frac=frac, dif_media=round(float((a-b).mean()), 2),
                        cohen_d=round(float(d), 2), p_wilcoxon=round(float(w.pvalue), 4), n=len(a)))
    except Exception as e:
        pass
if sup:
    sup7 = pd.DataFrame(sup); sup7.to_csv(OUT_DIR / "wilcoxon_suplementar_7k.csv", index=False)
    print("\n--- Suplemento (Wilcoxon unilateral entre seeds; piso de p = 0,03125 com n=5) ---")
    print(sup7.to_string(index=False))

In [ ]:
# ============================================================================
# 13.3 H2 no 7k: o gap encolhe sob dimensao casada?  (bloqueava o paper desde a Semana 2)
# ============================================================================
def gap(armA, armB, cond, frac):
    a = df7[(df7.arm==armA)&(df7.cond==cond)&(df7.frac==frac)]["bal_acc"].mean()
    b = df7[(df7.arm==armB)&(df7.cond==cond)&(df7.frac==frac)]["bal_acc"].mean()
    return a - b

print("=== H2 no 7K (o teste que faltava) ===")
print(f"{'par':32s} {'frac':>6s} {'RAW':>8s} {'PCA'+str(COMMON_D):>8s} {'delta':>8s}  veredito")
h2rows = []
pares = [("Phikon", CLASSICO_MANCHETE)]
if SSL_MANCHETE: pares.append((SSL_MANCHETE, CLASSICO_MANCHETE))
pares = [(a, b) for a, b in pares if a in PAIRS and b in PAIRS]
for A, B in pares:
    for frac in [0.05, 1.00]:
        gr = gap(A, B, "RAW", frac); gp = gap(A, B, f"PCA{COMMON_D}", frac)
        v = "gap CRESCE" if gp - gr > 0.5 else ("gap ENCOLHE" if gp - gr < -0.5 else "gap PLANO")
        print(f"{A+' - '+B:32s} {int(frac*100):5d}% {gr:8.2f} {gp:8.2f} {gp-gr:+8.2f}  {v}")
        h2rows.append(dict(A=A, B=B, frac=frac, raw=round(gr,2), pca=round(gp,2),
                           delta=round(gp-gr,2), veredito=v))
pd.DataFrame(h2rows).to_csv(OUT_DIR / "h2_no_7k.csv", index=False)
print("\nH2 pre-registrada dizia: casando d, o gap ENCOLHE.")
print("Interno (Semana 2, eixo Phikon): cresce. Interno (Semanas 3-4, eixo SSL): plano.")
print("Escrever a afirmacao FRACA e defensavel: 'casar dimensao nao restaura paridade'.")

ev = df7[(df7.cond==f"PCA{COMMON_D}") & (df7.frac==1.0)][["arm","dim","evr"]].dropna()
print("\nVariancia retida em d=%d (7k, frac=100%%):" % COMMON_D)
print(ev.assign(evr=(100*ev.evr).round(1)).to_string(index=False))

## 14. H4 no 7k — a decomposição por classe (pré-registrada)

Teste direcional, com as 6 classes nomeadas no §2. A estatística é bootstrap sobre o teste: reamostra-se o 7k, recalcula-se o gap por classe, e mede-se a diferença *(gap médio nas morfológicas) − (gap médio nas texturais)*. H4 prevê **positivo**.

Duas armadilhas que a célula evita: (i) BACK/DEB/LYM ficam **fora** do teste (entram só na tabela descritiva); (ii) o "clássico" do gap é o **clássico-manchete travado na célula 8.1**, não o melhor por classe.

In [ ]:
# ============================================================================
# 14. H4 no 7k
# ============================================================================
mi = [CLASSES.index(c) for c in H4_MORFO]
ti = [CLASSES.index(c) for c in H4_TEXTURAL]

def recalls(yp, idx=None):
    yt = y_7k if idx is None else y_7k[idx]
    yq = yp if idx is None else yp[idx]
    cm = confusion_matrix(yt, yq, labels=range(len(CLASSES)))
    return 100.0 * np.diag(cm) / np.maximum(cm.sum(1), 1)

def teste_h4(arm_aprendido, arm_classico, frac=1.0, cond="RAW"):
    ya = load_pred(arm_aprendido, cond, frac, LABEL_SEEDS[0])
    yc = load_pred(arm_classico,  cond, frac, LABEL_SEEDS[0])
    gap_pt = recalls(ya) - recalls(yc)
    obs = float(gap_pt[mi].mean() - gap_pt[ti].mean())
    boots = []
    for b in BOOT:
        g = recalls(ya, b) - recalls(yc, b)
        boots.append(g[mi].mean() - g[ti].mean())
    boots = np.array(boots)
    return gap_pt, obs, float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5)), float((boots <= 0).mean())

H4_TAB = {}
alvos = [("Phikon", CLASSICO_MANCHETE)]
if SSL_MANCHETE: alvos.append((SSL_MANCHETE, CLASSICO_MANCHETE))
alvos = [(a, b) for a, b in alvos if a in PAIRS and b in PAIRS]
if not alvos:
    print("[H4] nenhum par aprendido-vs-classico disponivel no 7k.")
for A, B in alvos:
    g, obs, lo, hi, p = teste_h4(A, B)
    H4_TAB[f"{A} - {B}"] = g
    print(f"\n=== H4: {A} - {B} (7k, frac=100%, RAW) ===")
    for c, v in zip(CLASSES, g):
        tipo = "morf" if c in H4_MORFO else ("text" if c in H4_TEXTURAL else "---")
        print(f"  {c:5s} [{tipo}] gap = {v:+7.2f} pt")
    print(f"  >> (morfologicas) - (texturais) = {obs:+.2f} pt   IC95% [{lo:+.2f}, {hi:+.2f}]   "
          f"p_boot(unilateral) = {p:.4f}")
    print(f"  >> H4 {'SUPORTADA' if lo > 0 else ('REFUTADA (sinal invertido)' if hi < 0 else 'NULA (IC cruza zero)')}")
    print("     Lembrete: BACK/DEB/LYM ficam FORA do teste (nao pre-registradas). Nao realocar agora.")

h4df = pd.DataFrame(H4_TAB, index=CLASSES)
h4df["tipo_H4"] = ["morfologica" if c in H4_MORFO else
                   ("textural" if c in H4_TEXTURAL else "nao-preregistrada") for c in CLASSES]
h4df.round(2).to_csv(OUT_DIR / "h4_gaps_7k.csv")
print("\n", h4df.round(2).to_string())

In [ ]:
# ============================================================================
# 14.1 Recall por classe e matrizes de confusao no 7k (figuras do paper)
# ============================================================================
BRACOS_FIG = [b for b in ["LBP", CLASSICO_MANCHETE, SSL_MANCHETE, "Phikon"] if b and b in PAIRS]
BRACOS_FIG = list(dict.fromkeys(BRACOS_FIG))
rec7, cm7 = {}, {}
for b in BRACOS_FIG:
    yp = load_pred(b, "RAW", 1.0, LABEL_SEEDS[0])
    rec7[b] = recalls(yp)
    cm7[b] = confusion_matrix(y_7k, yp, labels=range(len(CLASSES)), normalize="true")
pd.DataFrame(rec7, index=CLASSES).round(2).to_csv(OUT_DIR / "recall_por_classe_7k.csv")
print(pd.DataFrame(rec7, index=CLASSES).round(2).to_string())

print("\nMassa fora da diagonal STR<->MUS no 7k (cegueira a anisotropia):")
for b in [x for x in ["LBP", "LBP-HF-H", "LBP-ENR", CLASSICO_MANCHETE] if x in PAIRS]:
    cmb = confusion_matrix(y_7k, load_pred(b, "RAW", 1.0, LABEL_SEEDS[0]), labels=range(len(CLASSES)))
    ia, ib = CLASSES.index("STR"), CLASSES.index("MUS")
    m = 100.0 * (cmb[ia, ib] + cmb[ib, ia]) / max(cmb[ia].sum() + cmb[ib].sum(), 1)
    print(f"  {b:14s} {m:5.2f}%")

In [ ]:
# ============================================================================
# 15. H3-a: o que ainda pode ser dito (achado direcional, com limitacao declarada)
# ============================================================================
abl = [k for k in PAIRS if k.startswith("SimCLR_abl_")]
if abl and SSL_MANCHETE:
    print("=== H3-a no 7k: vantagem sobre o classico, full vs ablado ===")
    for frac in [0.05, 1.00]:
        full = df7[(df7.arm==SSL_MANCHETE)&(df7.cond=="RAW")&(df7.frac==frac)]["bal_acc"].mean()
        cls  = df7[(df7.arm==CLASSICO_MANCHETE)&(df7.cond=="RAW")&(df7.frac==frac)]["bal_acc"].mean()
        a    = np.mean([df7[(df7.arm==k)&(df7.cond=="RAW")&(df7.frac==frac)]["bal_acc"].mean() for k in abl])
        print(f"  frac {int(frac*100):3d}%: vantagem(full) = {full-cls:+6.2f}   "
              f"vantagem(ablado) = {a-cls:+6.2f}   queda = {(full-cls)-(a-cls):+6.2f}")
else:
    print("Sem braco ablado no 7k (checkpoints ausentes).")
print(
    "\nLIMITACAO OBRIGATORIA NO TEXTO (registro 3-4, secao 3 - nao negociavel):\n"
    "a ablacao removeu RandomResizedCrop E cor de uma vez, o que destruiu a APRENDIBILIDADE\n"
    "do pretexto (loss caindo enquanto o kNN-monitor caia = shortcut learning, cf. Chen et al.\n"
    "2020), nao apenas o CONTEUDO de invariancia. A queda full->ablado e consistente com H3-a\n"
    "mas NAO isola invariancia de colapso do pretexto. H3-a e achado direcional; nao pode\n"
    "sustentar a contribuicao central. Re-rodar a versao so-cor (mantendo o crop) ficou fora do\n"
    "orcamento computacional desta edicao - declarar como trabalho futuro, com o motivo.\n")

In [ ]:
# ============================================================================
# 16. Figuras do paper (PDF vetorial, tamanho de coluna IEEE)
# ============================================================================
plt.rcParams.update({"font.size": 8, "figure.dpi": 150, "savefig.bbox": "tight"})
COL = 3.4

# Fig 1 - curva de fracao de rotulo no 7k
fig, ax = plt.subplots(figsize=(COL, 2.4))
for b in [x for x in ["Fractal", "LBP", CLASSICO_MANCHETE, SSL_MANCHETE, "Phikon"] if x and x in PAIRS]:
    d = df7[(df7.arm == b) & (df7.cond == "RAW")].groupby("frac")["bal_acc"].mean()
    ax.plot([f*100 for f in d.index], d.values, marker="o", ms=3, label=b)
ax.set_xscale("log"); ax.set_xlabel("label fraction (%)"); ax.set_ylabel("balanced accuracy (%)")
ax.grid(alpha=.3); ax.legend(fontsize=6)
fig.savefig(FIG_DIR / "fig1_curva_rotulo_7k.pdf"); plt.close(fig)

# Fig 2 - gap por classe (H4)
if H4_TAB:
    k0 = list(H4_TAB)[0]
    g = H4_TAB[k0]
    ordem = np.argsort(-g)
    cores = ["#444444" if CLASSES[i] in H4_MORFO else
             ("#999999" if CLASSES[i] in H4_TEXTURAL else "#cccccc") for i in ordem]
    fig, ax = plt.subplots(figsize=(COL, 2.2))
    ax.bar([CLASSES[i] for i in ordem], g[ordem], color=cores)
    ax.axhline(0, lw=.8, c="k"); ax.set_ylabel(f"gap (pts)\n{k0}")
    ax.tick_params(axis="x", rotation=45); ax.grid(axis="y", alpha=.3)
    fig.savefig(FIG_DIR / "fig2_gap_por_classe_7k.pdf"); plt.close(fig)

# Fig 3 - matrizes de confusao
if cm7:
    fig, axes = plt.subplots(1, len(cm7), figsize=(2.0*len(cm7), 2.2))
    axes = np.atleast_1d(axes)
    for axi, (b, cm) in zip(axes, cm7.items()):
        axi.imshow(cm, vmin=0, vmax=1, cmap="Greys")
        axi.set_title(b, fontsize=7)
        axi.set_xticks(range(9)); axi.set_yticks(range(9))
        axi.set_xticklabels(CLASSES, rotation=90, fontsize=5)
        axi.set_yticklabels(CLASSES if axi is axes[0] else [], fontsize=5)
    fig.savefig(FIG_DIR / "fig3_confusao_7k.pdf"); plt.close(fig)

# Fig 4 - escada H3-b (o que o enriquecimento classico comprou)
esc = [x for x in ["LBP", "LBP-H", "LBP-HF-H", "LBP-COR-H", "LBP-ENR",
                   "Fractal", "Fractal-BM", "CLASSICO-MAX"] if x in PAIRS]
if esc:
    fig, ax = plt.subplots(figsize=(COL, 2.3))
    v = [df7[(df7.arm==b)&(df7.cond=="RAW")&(df7.frac==1.0)]["bal_acc"].mean() for b in esc]
    ax.barh(range(len(esc)), v, color="#666666")
    ax.set_yticks(range(len(esc))); ax.set_yticklabels(esc, fontsize=6)
    ax.set_xlabel("balanced accuracy (%) @ 100% labels, 7k"); ax.grid(axis="x", alpha=.3)
    fig.savefig(FIG_DIR / "fig4_escada_classica_7k.pdf"); plt.close(fig)

print("figuras em", FIG_DIR, ":", sorted(p.name for p in FIG_DIR.glob("*.pdf")))

In [ ]:
# ============================================================================
# 17. Tabelas em LaTeX (booktabs) para colar no template IEEE
# ============================================================================
def tex_tabela_principal():
    bracos = [x for x in ["Fractal", "Fractal-BM", "LBP", "LBP-ENR", CLASSICO_MANCHETE,
                          SSL_MANCHETE, "Phikon"] if x and x in PAIRS]
    bracos = list(dict.fromkeys(bracos))
    L = [r"\begin{table}[t]", r"\centering",
         r"\caption{Balanced accuracy (\%) on the external cohort CRC-VAL-HE-7K. "
         r"Probes trained on stratified subsamples of the NCT-CRC-HE-100K training pool. "
         r"Mean over 5 label seeds; 100\% is deterministic (single point).}",
         r"\label{tab:main}", r"\begin{tabular}{l r " + "r"*len(LABEL_FRACTIONS) + "}", r"\toprule",
         "Features & $d$ & " + " & ".join(f"{int(f*100)}\\%" for f in LABEL_FRACTIONS) + r" \\",
         r"\midrule"]
    for b in bracos:
        d = int(PAIRS[b][0].shape[1])
        vs = []
        for f in LABEL_FRACTIONS:
            s = df7[(df7.arm==b)&(df7.cond=="RAW")&(df7.frac==f)]["bal_acc"]
            vs.append(f"{s.mean():.2f}" + (f"\\,$\\pm$\\,{1.96*s.std(ddof=1)/np.sqrt(len(s)):.2f}"
                                           if len(s) > 1 else ""))
        L.append(f"{b} & {d} & " + " & ".join(vs) + r" \\")
    L += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    return "\n".join(L)

def tex_tabela_h4():
    if not H4_TAB: return ""
    k0 = list(H4_TAB)[0]; g = H4_TAB[k0]
    L = [r"\begin{table}[t]", r"\centering",
         r"\caption{Per-class recall gap (percentage points) on CRC-VAL-HE-7K, "
         r"100\% labels. Class groups (morphological vs.\ textural) were pre-registered; "
         r"BACK, DEB and LYM were not assigned a priori and are excluded from the test.}",
         r"\label{tab:h4}", r"\begin{tabular}{l l r}", r"\toprule",
         r"Class & Group & Gap (pts) \\", r"\midrule"]
    for c, v in zip(CLASSES, g):
        grp = "morphological" if c in H4_MORFO else ("textural" if c in H4_TEXTURAL else "---")
        L.append(f"{c} & {grp} & ${v:+.2f}$ " + r"\\")
    L += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    return "\n".join(L)

tex = tex_tabela_principal() + "\n\n" + tex_tabela_h4()
(OUT_DIR / "tabelas.tex").write_text(tex, encoding="utf-8")
print(tex[:1200])
print("\n[gravado em", OUT_DIR / "tabelas.tex", "]")

In [ ]:
# ============================================================================
# 18. Emissor do registro (registro_semana5_6.md)
# ============================================================================
def emit_registro():
    hoje = time.strftime("%d/%m/%Y")
    L = []
    A = L.append
    A(f"# Registro - Semanas 5-6 (WVC 2026, §11): fatorial-nucleo, H3-b, H4 e TOQUE UNICO no 7k\n\n")
    A(f"**Data do registro:** {hoje}\n\n")
    A("**Ramo tomado:** fechamento experimental. Semanas 3-4 nao re-rodadas -> tres decisoes "
      "fechadas para o lado conservador (H3-a rebaixada; SimCLR declarado budget-limited/nao-convergido; "
      "manchete SSL treinada em 80k).\n\n")
    A(f"**Split desta execucao:** train_pool={len(idx_train_pool)} / eval_int={len(idx_eval_int)} "
      f"(INTERNAL_SPLIT_SEED={INTERNAL_SPLIT_SEED}).\n\n")
    A(f"**Pre-commit do 7k:** hash `{PRECOMMIT['spec_hash']}`, gravado em `{PRECOMMIT_PATH.name}`.\n\n")

    A("## 1. Numeros do paper (7k, coorte externa, toque unico)\n\n")
    piv = df7[df7.cond=="RAW"].pivot_table(index="arm", columns="frac", values="bal_acc").round(2)
    A(piv.to_markdown() + "\n\n")
    A("Condicao PCA%d:\n\n" % COMMON_D)
    A(df7[df7.cond==f"PCA{COMMON_D}"].pivot_table(index="arm", columns="frac",
                                                  values="bal_acc").round(2).to_markdown() + "\n\n")

    A("## 2. Estatistica pareada (IC95% bootstrap sobre o teste)\n\n")
    A((stat7.to_markdown(index=False) if len(stat7) else "_sem comparacoes disponiveis_") + "\n\n")
    A("> **Desvio registrado do §7:** o Wilcoxon pareado sobre 5 seeds nao consegue p<0,05 "
      "(minimo bilateral = 0,0625). Substituido por diferenca pareada + IC bootstrap estratificado "
      "sobre os 7.180 patches + Holm. Valido tambem em frac=100%, onde nao ha seeds.\n\n")

    A("## 3. H2 no 7k (pendencia que bloqueava o paper desde a Semana 2)\n\n")
    A(pd.DataFrame(h2rows).to_markdown(index=False) + "\n\n")

    A("## 4. H3-b: o que o enriquecimento classico comprou\n\n")
    A(h3b_piv.round(2).to_markdown() + "\n\n")
    A(f"**Classico-manchete TRAVADO (antes do 7k):** `{CLASSICO_MANCHETE}`. "
      "Regra pre-declarada: maior media entre as fracoes; empate dentro de 0,5 pt resolvido pela "
      "menor dimensao.\n\n")
    A("> **Guarda de leitura:** se o gap encolheu, isso significa 'parte do gap era pobreza do "
      "descritor', NAO 'o classico alcanca o aprendido'. O paradigma nao foi testado; o descritor foi.\n\n")

    A("## 5. H4 (decomposicao por classe)\n\n")
    A(h4df.round(2).to_markdown() + "\n\n")
    A("> Pre-registro cobre 6 das 9 classes. BACK, DEB e LYM nao foram atribuidas a priori e "
      "ficam FORA do teste direcional.\n\n")

    A("## 6. Diagnostico de stain\n\n")
    A(f"- OD medio 100k: {STAIN_DIAG['od_100k']} | 7k: {STAIN_DIAG['od_7k']}\n")
    A(f"- |delta| razao cromatica: {STAIN_DIAG['delta_chroma']} | dispersao relativa: {STAIN_DIAG['disp_rel']}x\n")
    A("- Documental: o registro Zenodo 10.5281/zenodo.1214456 descreve NCT-CRC-HE-100K como "
      "normalizado por Macenko e distribui NCT-CRC-HE-100K-NONORM separadamente. Se NCT_DIR veio "
      "do primeiro, a condicao do nucleo (§3) bate com o declarado.\n\n")

    A("## 7. Guardas que sobrevivem para a escrita\n\n")
    A("- **(a)** Nenhum numero interno em tabela de desempenho. So o 7k.\n")
    A("- **(b)** H3-a e colapso de pretexto, nao ablacao de invariancia. Achado direcional + "
      "limitacao declarada. Nao sustenta a contribuicao central.\n")
    A("- **(c)** SimCLR subtreinado (parou por relogio na subida). O gap Phikon-SimCLR e "
      "LIMITE SUPERIOR. Proibido escrever 'SSL estaciona abaixo do FM'.\n")
    A("- **(d)** Reportar o PROBE, nunca o kNN-monitor, como qualidade do SimCLR.\n")
    A("- **(e)** H2: a afirmacao defensavel e a fraca - 'casar dimensao nao restaura paridade'. "
      "O gap CRESCE no eixo Phikon e fica PLANO no eixo SSL; o mecanismo de EVR difere.\n")
    A("- **(f)** Validacao assimetrica: so o Phikon tem ancora publicada (eva/kaiko, Tabela 3, 0,935).\n")
    A("- **(g)** H3-b move o lado classico, nao o aprendido.\n")
    A("- **(h)** Manchete SSL treinada em 80k (guarda transdutiva), nao 100k. Conservador; declarar.\n\n")

    A("## 8. Bibliografia - status\n\n")
    A("- **[FECHADO]** eva: kaiko.ai, I. Gatopoulos, N. Kanzig, R. Moser, S. Otalora, "
      "*eva: Evaluation framework for pathology foundation models*, MIDL 2024. Valor do Phikon-CRC "
      "na **Tabela 3** ('Linear probing evaluation of FMs, averaged balanced accuracy'): 0,935.\n")
    A("- **[FECHADO]** Dataset: Kather, Halama, Marx (2018), Zenodo 10.5281/zenodo.1214456.\n")
    A("- **[a conferir]** Ahonen et al. 2009 (LBP-HF); Kather et al. 2016 (Sci. Rep. 6:27988); "
      "Sarkar & Chaudhuri (DBC); Allain & Cloitre (lacunaridade); Backes & Bruno "
      "(Bouligand-Minkowski); Chen et al. 2020 (SimCLR); Filiot et al. (Phikon); "
      "Macenko et al. 2009 (ISBI, DOI 10.1109/ISBI.2009.5193250).\n\n")

    A("## 9. Itens abertos ao fim das Semanas 5-6\n\n")
    A("- Escrita (Semana 7): 6 paginas IEEE, titulo e resumo em ingles, checklist duplo-cego (§12).\n")
    A("- Trabalho futuro declarado: ablacao H3-a so-cor (mantendo crop); SimCLR convergido; "
      "pre-treino em 100k para a manchete.\n")
    A("- Conformidade: todo enquadramento interno fica FORA do PDF.\n")
    txt = "".join(L)
    (OUT_DIR / "registro_semana5_6.md").write_text(txt, encoding="utf-8")
    print(txt[:2500]); print("\n...\n[gravado em", OUT_DIR / "registro_semana5_6.md", "]")

emit_registro()

---
## Fecho — o que sai daqui e o que a escrita não pode dizer

**Sai daqui:** `resultados_7k_FINAL.csv` (a tabela de desempenho do paper), `estatistica_pareada_7k.csv`, `h2_no_7k.csv`, `h4_gaps_7k.csv`, `h3b_escada_classica_interno.csv`, `recall_por_classe_7k.csv`, quatro figuras em PDF vetorial, `tabelas.tex` e `registro_semana5_6.md`.

**A escrita não pode dizer:**

- que H3-a isolou invariância — ela conflacionou invariância com colapso do pretexto;
- que o SSL in-domain estaciona abaixo do FM — ele nunca parou de subir;
- que o gap encolhe sob dimensão casada — a afirmação defensável é a fraca, *"casar dimensão não restaura paridade"*, e o mecanismo difere entre Phikon e SimCLR;
- que o clássico enriquecido testou o paradigma clássico — ele testou **estes** descritores, neste orçamento;
- nada baseado em número interno.

**Ordem de escrita sugerida, dado o prazo:** Métodos primeiro (está todo travado nos registros e não depende de mais nenhum resultado), depois Resultados a partir dos CSVs, depois Discussão, e a Introdução por último — ela é a única seção que muda de acordo com o que os números disseram.

**Checklist §12 antes de submeter:** título e resumo em inglês (resumo ≤150 palavras), ≤6 páginas, template IEEE, **duplo-cego** (sem nome, afiliação, grant, link ou agradecimento identificável — inclusive nos metadados do PDF), conflitos no CMT, submissão até 24/08/2026.